# 【Colab版】週末のシグナル確認：SOMPO ホールディングス（8630）／ウィリアムズ%R(14) < -80

**このファイル1つで動く Google Colab 用の版です。** 計算に使うプログラムと銘柄一覧（TOPIX500 相当、2026/9/30 時点の JPX 一覧）を、下の「0. 準備」セルに埋め込んであります。

### 使い方
1. [Google Colab](https://colab.research.google.com/) を開き、「ファイル」→「ノートブックをアップロード」でこのファイルを選ぶ
2. 金曜の取引終了後〜日曜に、メニューの「ランタイム」→「すべてのセルを実行」
3. 「3. 今週末の判定」に、シグナルの有無と同時に出ている条件が表示される

- 毎回、全約490銘柄の株価を Yahoo Finance から取り直すため、全体で5〜10分ほどかかります（Colab はセッションが終わるとデータが消えるため）。
- 「0. 準備」セルは変更しないでください。設定は「1. 設定」で変えられます。

---

毎週末（金曜の取引終了後〜日曜）に上から順に実行し、**SOMPO にシグナルが出ているか**を確認するノートブックです。あわせて、**上昇確率を上げる・下げる条件が同時に出ているか**を確認し、その条件が過去に本当に効いていたかを検証できます。

**判定の型（これまでの検証と同じ）**
- 週の最終営業日の終値で **ウィリアムズ%R(14) < -80** なら「シグナルあり」
- 翌週の最初の営業日の寄りで買い、最長でその週の最終営業日の引けで売る

**分析時点（2008〜2026/10）の結果**
- シグナル172回、翌週の上昇確率64.0%
- 片道0.1%のコスト控除後で、1回あたり平均 +0.84%
- 信用50万円・複利で2011/10〜2026/10 に仕掛けた場合の試算：損切りなし +78万円、損切り7% +40万円（税引前）

**同時に見る条件**

| 向き | 条件 | 根拠 |
|---|---|---|
| ＋ | 陽線3日連続、25日線乖離率 ≤ -10%、パラボリックSAR買い転換、市場全体が週間 -3% 以下 | 全銘柄で前半・後半とも上昇確率が高かった |
| － | 終値 > 75日線、市場全体が75日線より上 | 全銘柄・他の保険株で上昇確率が低かった |
| ？ | 浅い下げ（週間 -3% 未満）、市場は上昇でこの銘柄だけ下落、業種ぐるみの下げ、この銘柄だけの下げ | SOMPO だけで目立った（偶然の可能性が高い） |

**参考点** ＝ ＋の条件の数 － －の条件の数（？は数えない）

> ⚠️ **注意**
> - SOMPO は、過去の成績が上位だったから選んだ銘柄です。多数の銘柄を比べたことを補正すると偶然と区別できず、実際より良く見えている可能性が高いです。
> - 条件も参考点も、売買ルールとして確かめたものではありません。
> - データは Yahoo Finance（非公式）です。対象は今上場している銘柄だけなので、成績が良く見えやすい偏り（生存者バイアス）があります。決算発表・権利落ちなどの予定は考慮していません。
> - このノートブックは判断材料をそろえるもので、売買の推奨ではありません。

## 0. 準備

In [ ]:
# 0. 準備（このセルは変更しないでください）
# 計算部分のプログラムと銘柄一覧をこの環境に書き出し、必要なライブラリと日本語フォントを用意する。
import subprocess
import sys
from pathlib import Path

LIB = Path("weekpick_colab")
SOURCES = {'weekpick/__init__.py': '"""日本株の週次銘柄評点ツール（金曜終値で採点 → 翌週月曜寄りで買い → 金曜引けで売り）。"""\n', 'weekpick/universe.py': '"""対象銘柄（ユニバース）。\n\n一次情報: 日本取引所グループ「東証上場銘柄一覧」（毎月更新）\n  https://www.jpx.co.jp/markets/statistics-equities/misc/01.html\n規模区分（TOPIX Core30 / Large70 / Mid400 / Small 1 / Small 2）で範囲を選ぶ。\n\n注意: 一覧は「取得時点で上場している銘柄」の「取得時点の規模区分」。過去の検証に使うと、\nその後に上場廃止・指数除外された銘柄が入らないため、成績が実際より良く見えやすい（生存者バイアス）。\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport pandas as pd\n\nJPX_LIST_URL = "https://www.jpx.co.jp/markets/statistics-equities/misc/tvdivq0000001vg2-att/data_j.xlsx"\nUNIVERSE_CSV = Path(__file__).resolve().parent.parent / "data" / "universe_jpx.csv"\n\nSCOPES: dict[str, list[str]] = {\n    "topix100": ["TOPIX Core30", "TOPIX Large70"],\n    "topix500": ["TOPIX Core30", "TOPIX Large70", "TOPIX Mid400"],\n    "topix1000": ["TOPIX Core30", "TOPIX Large70", "TOPIX Mid400", "TOPIX Small 1"],\n}\n\n\ndef fetch_jpx_list(out: Path = UNIVERSE_CSV) -> pd.DataFrame:\n    """JPXの上場銘柄一覧を取得し、内国株式だけをCSVに保存する。"""\n    import io\n    import urllib.request\n\n    req = urllib.request.Request(JPX_LIST_URL, headers={"User-Agent": "Mozilla/5.0"})\n    with urllib.request.urlopen(req, timeout=60) as r:\n        raw = pd.read_excel(io.BytesIO(r.read()), dtype=str)\n    df = raw.rename(columns={\n        "日付": "list_date", "コード": "code", "銘柄名": "name", "市場・商品区分": "market",\n        "33業種区分": "sector", "規模区分": "size",\n    })[["list_date", "code", "name", "market", "sector", "size"]]\n    df = df[df["market"].str.contains("内国株式", na=False)].reset_index(drop=True)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(out, index=False, encoding="utf-8")\n    return df\n\n\ndef load_universe(scope: str = "topix500", path: Path = UNIVERSE_CSV) -> pd.DataFrame:\n    """scope: topix100 / topix500 / topix1000 / prime（プライム全銘柄） / ティッカー列挙CSVのパス。\n\n    戻り値の列: ticker（例 7203.T）, code, name, sector, size\n    """\n    p = Path(scope)\n    if p.suffix == ".csv" and p.exists():\n        df = pd.read_csv(p, dtype=str)\n        if "ticker" not in df.columns:\n            df["ticker"] = df["code"].str.strip() + ".T"\n        return df\n    if not path.exists():\n        raise FileNotFoundError(f"{path} がありません。先に `python -m weekpick universe` を実行してください")\n    df = pd.read_csv(path, dtype=str)\n    if scope == "prime":\n        df = df[df["market"].str.startswith("プライム")]\n    elif scope in SCOPES:\n        df = df[df["size"].isin(SCOPES[scope])]\n    else:\n        raise ValueError(f"未知のユニバース: {scope}（{\', \'.join([*SCOPES, \'prime\'])} または CSV パス）")\n    df = df.copy()\n    df["ticker"] = df["code"].str.strip() + ".T"\n    return df.reset_index(drop=True)\n', 'weekpick/data.py': '"""複数銘柄の日足を yfinance でまとめて取得し、銘柄×日付の表（パネル）にする。\n\nyfinance（非公式API）の値の扱い:\n  - open/high/low/close/volume は株式分割を調整済み、配当は未調整\n  - adj_close は配当も調整した終値。adjf = adj_close / close を「配当込み係数」として使い、\n    週の損益に配当（権利落ちによる値下がり分）を含める\n"""\nfrom __future__ import annotations\n\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom backtest225.data import clean\n\nCACHE_DIR = Path(__file__).resolve().parent.parent / "data" / "cache" / "jp"\nFIELDS = ["open", "high", "low", "close", "volume", "adjf"]\n\n\ndef _cache_path(ticker: str) -> Path:\n    return CACHE_DIR / f"{ticker.replace(\'.\', \'_\')}.csv"\n\n\ndef fetch(tickers: list[str], refresh: bool = False, chunk: int = 50, start: str = "2005-01-01") -> list[str]:\n    """未取得（refresh時は全部）の銘柄を取得してキャッシュに保存。取得できなかった銘柄を返す。"""\n    import yfinance as yf\n\n    CACHE_DIR.mkdir(parents=True, exist_ok=True)\n    todo = [t for t in tickers if refresh or not _cache_path(t).exists()]\n    failed: list[str] = []\n    for i in range(0, len(todo), chunk):\n        part = todo[i : i + chunk]\n        print(f"  取得中 {i + 1}〜{i + len(part)} / {len(todo)}", flush=True)\n        raw = yf.download(part, start=start, auto_adjust=False, progress=False, actions=False,\n                          group_by="ticker", threads=True)\n        for t in part:\n            try:\n                d = raw[t] if isinstance(raw.columns, pd.MultiIndex) else raw\n                d = d.rename(columns=str.lower).dropna(subset=["close"])\n            except KeyError:\n                d = pd.DataFrame()\n            if d.empty:\n                failed.append(t)\n                continue\n            out = d[["open", "high", "low", "close", "volume"]].copy()\n            out["adj_close"] = d["adj close"]\n            out.index = pd.to_datetime(out.index).tz_localize(None)\n            out.index.name = "date"\n            out.to_csv(_cache_path(t))\n        time.sleep(1)  # 連続アクセスを控えめに\n    return failed\n\n\n@dataclass\nclass Panel:\n    """各フィールドが「行=日付, 列=ティッカー」の DataFrame。"""\n\n    open: pd.DataFrame\n    high: pd.DataFrame\n    low: pd.DataFrame\n    close: pd.DataFrame\n    volume: pd.DataFrame\n    adjf: pd.DataFrame\n    notes: list[str]\n\n    @property\n    def tickers(self) -> list[str]:\n        return list(self.close.columns)\n\n\nSPLIT_RATIOS = [2, 3, 4, 5, 10, 20, 25, 50, 100]\n\n\ndef adjust_unadjusted_splits(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:\n    """前後5営業日の終値の中央値が、ほぼ整数倍（2,3,4,5,10…）で段差になっている日を未調整の分割\n    （または併合）とみなし、前後5日が平穏（日々の値動き10%未満）な場合に限ってそれより前の価格を割る（出来高は掛ける）。Yahoo が分割を反映し損ねる場合の対策。\n    """\n    df = df.copy()\n    found: list[str] = []\n    after_date = df.index[0]\n    while True:\n        c = df["close"]\n        before = c.rolling(5, min_periods=3).median().shift(1)\n        after = c[::-1].rolling(5, min_periods=3).median()[::-1]\n        prev = c.shift(1)\n        move = (c / prev - 1).abs()\n        calm = (move.shift(1).rolling(5, min_periods=3).max() < 0.1) & (move[::-1].shift(1).rolling(5, min_periods=3).max()[::-1] < 0.1)\n        hit = None\n        for d, x in (before / after).dropna().items():\n            if d <= after_date:\n                continue\n            inv = x >= 1\n            k = x if inv else 1 / x\n            best = min(SPLIT_RATIOS, key=lambda m: abs(k / m - 1))\n            day = prev[d] / c[d]  # 当日の段差も同じ大きさか\n            if calm[d] and k >= 1.8 and abs(k / best - 1) <= 0.06 and abs((day if inv else 1 / day) / best - 1) < 0.15:\n                hit = (d, best, inv)\n                break\n        if hit is None:\n            break\n        d, best, inv = hit\n        f = best if inv else 1 / best\n        pre = df.index < d\n        df.loc[pre, ["open", "high", "low", "close"]] /= f\n        df.loc[pre, "volume"] *= f\n        found.append(f"{d.date()} {\'1:\' if inv else \'\'}{best:g}{\'\' if inv else \':1\'}")\n        after_date = d\n    return df, found\n\n\ndef drop_outliers(df: pd.DataFrame, ratio: float = 3.0, min_ref: float = 100.0) -> tuple[pd.DataFrame, int]:\n    """直前の正常な終値（min_ref 円以上）から ratio 倍超・1/ratio 未満に飛んだ日を欠損にする。\n\n    東証の値幅制限では、100円以上の株価が1日で3倍・3分の1になることはないため、配信データの誤り\n    （上場廃止前後の異常値など）とみなす。100円未満の銘柄の暴落（例: 経営破綻時）は本物の値動きとして残す。\n    30日以上データが途切れた後（再上場など）は基準を取り直す。\n    """\n    close = df["close"].to_numpy()\n    idx = df.index\n    bad = np.zeros(len(df), dtype=bool)\n    ref, ref_date = np.nan, None\n    for i, x in enumerate(close):\n        if np.isnan(x):\n            continue\n        if ref_date is not None and (idx[i] - ref_date).days > 30:\n            ref = np.nan\n        if ref >= min_ref and (x / ref > ratio or x / ref < 1 / ratio):\n            bad[i] = True\n            continue\n        ref, ref_date = x, idx[i]\n    if bad.any():\n        df = df.copy()\n        df.loc[bad] = np.nan\n    return df, int(bad.sum())\n\n\ndef dividend_factor(adj: pd.Series, close: pd.Series, max_yield: float = 0.2) -> tuple[pd.Series, int]:\n    """配当込み係数（最新日=1）。adj_close / close の日々の変化のうち、配当として妥当なもの\n    （権利落ち日に 0〜max_yield 相当だけ増える）だけを残す。Yahoo の adj_close は銘柄により\n    桁外れ・負の値になることがあるため。2つ目の戻り値は異常を無視したかどうか（0/1）。\n    """\n    f = (adj / close).replace([np.inf, -np.inf], np.nan).ffill().bfill()\n    g = (f / f.shift(1)).fillna(1.0)\n    ok = (g >= 1 - 1e-6) & (g <= 1 / (1 - max_yield))\n    bad = int((~ok & ((g - 1).abs() > 1e-6)).any())\n    g = g.where(ok, 1.0).clip(lower=1.0)\n    cum = g.cumprod()\n    return cum / cum.iloc[-1], bad\n\n\ndef build_panel(frames: dict[str, pd.DataFrame], min_coverage: float = 0.3) -> Panel:\n    """銘柄ごとの日足（open..volume, adj_close）からパネルを作る。\n\n    営業日は「全銘柄の半数近くが値を持つ日」とし、祝日に紛れ込んだ単独の行を除く。\n    """\n    notes: list[str] = []\n    fixed = broken = 0\n    dropped: list[str] = []\n    splits: list[str] = []\n    cols: dict[str, dict[str, pd.Series]] = {f: {} for f in FIELDS}\n    for t, df in frames.items():\n        adj = df["adj_close"] if "adj_close" in df.columns else df["close"]\n        c, n = clean(df[["open", "high", "low", "close", "volume"]])\n        fixed += sum(1 for x in n if "補正" in x or "代用" in x)\n        c = c.where(c["volume"] > 0)  # 出来高0の日は約定がなく、配信値は埋め草のことがある（合併前後など）\n        c, sp = adjust_unadjusted_splits(c)\n        if sp:\n            splits.append(f"{t}({\', \'.join(sp)})")\n        c, k = drop_outliers(c)\n        if k:\n            dropped.append(f"{t}({k}日)")\n        f, bad = dividend_factor(adj.reindex(c.index), df["close"].reindex(c.index))\n        broken += bad\n        for k in ["open", "high", "low", "close", "volume"]:\n            cols[k][t] = c[k]\n        cols["adjf"][t] = f\n    wide = {k: pd.DataFrame(v).sort_index() for k, v in cols.items()}\n    count = wide["close"].notna().sum(axis=1)\n    days = count.index[count >= count.rolling(20, min_periods=1).max() * min_coverage]\n    wide = {k: v.reindex(days) for k, v in wide.items()}\n    if splits:\n        notes.append("未調整の株式分割とみられる段差を検出し、それ以前の価格を調整: " + ", ".join(splits))\n    if dropped:\n        notes.append("直前の正常な終値から1日で3倍超/3分の1未満に飛んだ日を除外: " + ", ".join(dropped))\n    if broken:\n        notes.append(f"配当込み終値に異常な変化があった {broken} 銘柄では、その日の配当調整を無視しました")\n    if fixed:\n        notes.append(f"{fixed} 銘柄でデータの欠損/桁ずれを補正しました（backtest225.data.clean と同じ処理）")\n    return Panel(**wide, notes=notes)\n\n\ndef load_panel(tickers: list[str]) -> Panel:\n    frames, missing = {}, []\n    for t in tickers:\n        p = _cache_path(t)\n        if not p.exists():\n            missing.append(t)\n            continue\n        frames[t] = pd.read_csv(p, index_col=0, parse_dates=True)\n    if not frames:\n        raise FileNotFoundError("価格データがありません。先に `python -m weekpick fetch <設定>` を実行してください")\n    panel = build_panel(frames)\n    if missing:\n        panel.notes.append(f"価格データがない {len(missing)} 銘柄を除外: {\', \'.join(missing[:10])}"\n                           + (" ほか" if len(missing) > 10 else ""))\n    return panel\n', 'weekpick/factors.py': '"""評点に使う指標（すべて「その日の終値までの情報」だけで計算）と、合計スコア。\n\n各指標は「値が大きいほど翌週に上がりやすい」と想定した向きにそろえてある。\n本当にその向きで効くかは `python -m weekpick factors` で確かめること。\n"""\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom .data import Panel\n\n# 名前 → 説明（レポート表示用）\nFACTORS: dict[str, str] = {\n    "reversal": "短期リバーサル: 直近5営業日の騰落率×(-1)（下げた銘柄ほど高い）",\n    "momentum": "中期モメンタム: 12か月前→1か月前の騰落率（直近1か月は除く）",\n    "high52": "52週高値への近さ: 終値 ÷ 過去250営業日の高値",\n    "volume_surge": "出来高急増: 5日平均出来高 ÷ 60日平均出来高",\n    "low_vol": "低ボラティリティ: 60日の日次騰落率の標準偏差×(-1)",\n}\n\n\ndef compute_factors(p: Panel) -> dict[str, pd.DataFrame]:\n    ac = p.close * p.adjf  # 配当込みの終値\n    ah = p.high * p.adjf\n    vol = p.volume.where(p.volume > 0)\n    return {\n        "reversal": -(ac / ac.shift(5) - 1),\n        "momentum": ac.shift(21) / ac.shift(252) - 1,\n        "high52": ac / ah.rolling(250, min_periods=200).max(),\n        "volume_surge": vol.rolling(5, min_periods=3).mean() / vol.rolling(60, min_periods=40).mean(),\n        "low_vol": -ac.pct_change(fill_method=None).rolling(60, min_periods=40).std(),\n    }\n\n\ndef eligible(p: Panel, min_turnover: float = 5e8, min_price: float = 0.0, min_history: int = 260) -> pd.DataFrame:\n    """足切り: 上場からの営業日数・直近20日の平均売買代金（円）・株価。"""\n    turnover = (p.close * p.volume).rolling(20, min_periods=15).mean()\n    history = p.close.notna().cumsum()\n    return p.close.notna() & (history >= min_history) & (turnover >= min_turnover) & (p.close >= min_price)\n\n\ndef score(factors: dict[str, pd.DataFrame], weights: dict[str, float], mask: pd.DataFrame) -> pd.DataFrame:\n    """日ごとに、足切りを通った銘柄の中での順位（0〜1）を重み付きで平均した値。\n\n    重みが0でない指標がひとつでも欠けている銘柄は対象外（NaN）。\n    """\n    used = {k: w for k, w in weights.items() if w}\n    unknown = set(used) - set(factors)\n    if unknown:\n        raise ValueError(f"未知の指標: {sorted(unknown)}（使えるもの: {\', \'.join(FACTORS)}）")\n    if not used:\n        raise ValueError("weights がすべて0です")\n    total = None\n    ok = mask.copy()\n    for k, w in used.items():\n        f = factors[k].where(mask)\n        ok &= f.notna()\n        r = f.rank(axis=1, pct=True) * w\n        total = r if total is None else total + r\n    return (total / sum(abs(w) for w in used.values())).where(ok)\n', 'weekpick/backtest.py': '"""週次の検証: 第k週の最終営業日の終値で採点 → 第k+1週の最初の営業日の始値で買い → 同じ週の最終営業日の終値で売る。\n\n週は月曜〜日曜で区切る（祝日で月曜が休みなら火曜の始値、金曜が休みなら木曜の終値）。\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nimport numpy as np\nimport pandas as pd\n\nfrom .data import Panel\nfrom .factors import compute_factors, eligible, score\n\nLOT = 100  # 国内株式の売買単位（2018/10以降すべて100株に統一。それ以前の銘柄も100株として近似）\n\n\n@dataclass\nclass Week:\n    signal: pd.Timestamp  # 採点日（前週の最終営業日）\n    entry: pd.Timestamp   # 買う日（その週の最初の営業日、始値）\n    exit: pd.Timestamp    # 売る日（その週の最終営業日、終値）\n\n\ndef weeks(index: pd.DatetimeIndex) -> list[Week]:\n    per = index.to_period("W-SUN")\n    g = pd.Series(index, index=index).groupby(per)\n    first, last = g.min(), g.max()\n    out = []\n    for k in range(1, len(first)):\n        out.append(Week(last.iloc[k - 1], first.iloc[k], last.iloc[k]))\n    return out\n\n\ndef week_returns(p: Panel, ws: list[Week], cost: float) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """各週・各銘柄の「月曜始値→金曜終値」の騰落率（配当込み・往復コスト控除後）と、買値（未調整の始値）。\n\n    買う日に出来高0・始値なしの銘柄は買えないので NaN。売る日に値がなければ週内の最後の終値で売ったとみなす。\n    """\n    close_ff = (p.close * p.adjf).ffill(limit=4)\n    rows, opens = {}, {}\n    for w in ws:\n        o = p.open.loc[w.entry].where(p.volume.loc[w.entry] > 0)\n        buy = o * p.adjf.loc[w.entry]\n        sell = close_ff.loc[w.exit]\n        gross = sell / buy - 1\n        rows[w.entry] = (1 + gross) * (1 - cost) / (1 + cost) - 1\n        opens[w.entry] = o\n    return pd.DataFrame(rows).T, pd.DataFrame(opens).T\n\n\n@dataclass\nclass Setup:\n    weights: dict[str, float]\n    min_turnover: float = 5e8\n    min_price: float = 0.0\n    min_history: int = 260\n    cost: float = 0.001  # 片道（手数料率+滑り）\n    capital: float | None = None  # 指定時: 100株単位・資金内で買える銘柄だけ、端数は現金\n    start: str | None = None\n    end: str | None = None\n\n\n@dataclass\nclass Prepared:\n    weeks: list[Week]\n    scores: pd.DataFrame     # 行=買う日（entry）, 列=銘柄\n    returns: pd.DataFrame    # 行=買う日, 列=銘柄（コスト控除後）\n    entry_open: pd.DataFrame\n    signal_close: pd.DataFrame\n    factors: dict[str, pd.DataFrame] = field(default_factory=dict)  # 行=買う日\n    mask: pd.DataFrame | None = None\n\n\ndef prepare(p: Panel, s: Setup) -> Prepared:\n    facs = compute_factors(p)\n    mask = eligible(p, s.min_turnover, s.min_price, s.min_history)\n    sc = score(facs, s.weights, mask)\n    ws = weeks(p.close.index)\n    if s.start:\n        ws = [w for w in ws if w.entry >= pd.Timestamp(s.start)]\n    if s.end:\n        ws = [w for w in ws if w.exit <= pd.Timestamp(s.end)]\n    if not ws:\n        raise ValueError("検証期間に週がありません")\n    ret, op = week_returns(p, ws, s.cost)\n    sig = [w.signal for w in ws]\n    ent = pd.DatetimeIndex([w.entry for w in ws])\n\n    def by_entry(df):\n        out = df.loc[sig]\n        out.index = ent\n        return out\n\n    return Prepared(ws, by_entry(sc), ret, op, by_entry(p.close),\n                    {k: by_entry(v) for k, v in facs.items()}, by_entry(mask))\n\n\ndef run(prep: Prepared, n: int, capital: float | None = None) -> pd.DataFrame:\n    """上位 n 銘柄を均等に買った場合の週次成績。\n\n    列: ret（ポートフォリオ）, universe（足切り通過・買える全銘柄の均等平均＝比較対象）, held（保有数）,\n        invested（投資比率）, picks（銘柄）\n    """\n    rows = []\n    for d in prep.scores.index:\n        r = prep.returns.loc[d]\n        tradable = r.notna()\n        sc = prep.scores.loc[d].where(tradable).dropna()\n        univ = r[sc.index].mean() if len(sc) else np.nan\n        budget = capital / n if capital else None\n        if budget:\n            sc = sc[prep.signal_close.loc[d, sc.index] * LOT <= budget]\n        picks = sc.nlargest(n).index\n        if budget:\n            op = prep.entry_open.loc[d, picks]\n            shares = np.floor(budget / (op * LOT)) * LOT\n            cost_in = shares * op\n            pnl = float((cost_in * r[picks]).sum())\n            ret, invested = pnl / capital, float(cost_in.sum()) / capital\n        else:\n            ret = float(r[picks].sum()) / n if len(picks) else 0.0\n            invested = len(picks) / n\n        rows.append({"date": d, "ret": ret, "universe": univ, "held": len(picks),\n                     "invested": invested, "picks": " ".join(picks)})\n    return pd.DataFrame(rows).set_index("date")\n\n\ndef stats(w: pd.DataFrame) -> dict:\n    """週次成績の要約。年率は 52週で換算。"""\n    r, u = w["ret"], w["universe"]\n    ex = (r - u).dropna()\n    eq = (1 + r).cumprod()\n    years = len(r) / 52\n    return {\n        "weeks": len(r),\n        "mean": r.mean(),\n        "median": r.median(),\n        "win_rate": (r > 0).mean(),\n        "cagr": eq.iloc[-1] ** (1 / years) - 1 if years > 0 and eq.iloc[-1] > 0 else np.nan,\n        "vol": r.std() * np.sqrt(52),\n        "sharpe": r.mean() / r.std() * np.sqrt(52) if r.std() > 0 else np.nan,\n        "max_dd": float((eq / eq.cummax() - 1).min()),\n        "worst_week": r.min(),\n        "excess_mean": ex.mean(),\n        "excess_t": ex.mean() / (ex.std() / np.sqrt(len(ex))) if len(ex) > 1 and ex.std() > 0 else np.nan,\n        "beat_rate": (ex > 0).mean(),\n        "avg_held": w["held"].mean(),\n        "invested": w["invested"].mean(),\n    }\n\n\ndef universe_stats(w: pd.DataFrame) -> dict:\n    """比較対象（足切り通過銘柄の均等平均）を毎週持った場合。"""\n    return stats(pd.DataFrame({"ret": w["universe"], "universe": w["universe"], "held": np.nan, "invested": 1.0}))\n\n\ndef factor_report(prep: Prepared, quantiles: int = 5) -> pd.DataFrame:\n    """指標ごとに、毎週の対象銘柄を指標の大きさで5分位に分け、翌週の平均騰落率（週平均との差）を比べる。\n\n    IC は指標と翌週騰落率の順位相関（週ごと）の平均。t値は週ごとの値のばらつきから計算。\n    """\n    out = []\n    for name, f in prep.factors.items():\n        qrets, ics = [], []\n        for d in f.index:\n            r = prep.returns.loc[d]\n            x = f.loc[d].where(prep.mask.loc[d])\n            ok = x.notna() & r.notna()\n            if ok.sum() < quantiles * 5:\n                continue\n            x, y = x[ok], r[ok]\n            q = pd.qcut(x.rank(method="first"), quantiles, labels=False)\n            qrets.append((y.groupby(q).mean() - y.mean()).values)\n            ics.append(x.rank().corr(y.rank()))\n        if not qrets:\n            continue\n        qr, ic = np.array(qrets), pd.Series(ics)\n        spread = qr[:, -1] - qr[:, 0]\n        row = {"factor": name, "weeks": len(qr)}\n        row.update({f"Q{i + 1}": qr[:, i].mean() for i in range(quantiles)})\n        row["Q5-Q1" if quantiles == 5 else "top-bottom"] = spread.mean()\n        row["t"] = spread.mean() / (spread.std() / np.sqrt(len(spread)))\n        row["IC"] = ic.mean()\n        row["IC_t"] = ic.mean() / (ic.std() / np.sqrt(len(ic)))\n        out.append(row)\n    return pd.DataFrame(out)\n', 'weekpick/timing.py': '"""特定銘柄の週末の買いタイミング判定（シグナルと、同時に出ている条件）。\n\n判定の型は weekpick の検証と同じ: 週の最終営業日の終値で判定 → 翌週初日の寄りで買い → その週の最終営業日の引けで売り。\nシグナルは SIGNALS（RSI(2)<10 / ウィリアムズ%R(14)<-80）から選び、同時に見る条件の組は\nCONDITIONS（RSI(2)・ANA/あおぞら銀行向け）または WR_CONDITIONS（%R・SOMPO 向け）を使う。\n条件の選び方は 2008〜2026年の検証に基づくが、いずれも統計的に確かとは言えないため、判定は「参考」として扱う。\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport pandas as pd\n\nfrom .backtest import weeks, week_returns\nfrom .data import Panel\nfrom .factors import eligible\n\nTARGETS = {"9202.T": "ANAホールディングス", "8304.T": "あおぞら銀行"}\nRSI2_THRESHOLD = 10.0\nWR_THRESHOLD = -80.0\n\n\n@dataclass(frozen=True)\nclass Signal:\n    key: str      # indicators() の値の名前\n    label: str\n    column: str   # 履歴・判定表での列名\n    threshold: float\n\n    def on(self, value) -> bool:\n        return bool(value < self.threshold)\n\n\nSIGNALS = {\n    "rsi2": Signal("rsi2", "RSI(2) < 10", "RSI2", RSI2_THRESHOLD),\n    "wr14": Signal("wr14", "ウィリアムズ%R(14) < -80", "%R14", WR_THRESHOLD),\n}\n\n\n@dataclass(frozen=True)\nclass Condition:\n    key: str\n    label: str\n    side: str  # "+": 上昇確率が上がる側 / "-": 下がる側 / "?": 1銘柄だけで目立った（偶然の可能性が高い）\n    note: str\n\n\nCONDITIONS = [\n    Condition("dev25", "25日線乖離率 ≤ -10%", "+", "2銘柄・全銘柄とも上昇確率が高い。全銘柄では前半・後半とも+3.7ポイント"),\n    Condition("wk10", "週間騰落率 ≤ -10%", "+", "2銘柄・全銘柄とも高い。全銘柄では効果が主に2018年以降"),\n    Condition("rsi14", "RSI(14) < 30", "+", "2銘柄で約80%。全銘柄では+1.2ポイントと小さい"),\n    Condition("mkt_drop", "市場全体（全銘柄平均）が週間-3%以下", "+", "全銘柄の効果は2018年以降に限られる"),\n    Condition("ma75_up", "75日線が上向き（20日前比）", "-", "2銘柄・全銘柄とも上昇確率が低い（上昇トレンド中の軽い押し目）"),\n    Condition("above200", "終値 > 200日線", "-", "あおぞらで大きく低下。ANA・全銘柄では差が小さい"),\n    Condition("mkt_up", "市場全体は週間で上昇（この銘柄だけ下落）", "-", "ANA・全銘柄で低下。あおぞらでは逆"),\n    Condition("sector_drop", "業種平均より週間で5%以上下落", "?", "あおぞらで17回中17回上昇。1銘柄だけの結果で偶然の可能性"),\n    Condition("down4", "4日以上連続で下落", "?", "あおぞらで83.8%。ANA・全銘柄では差なし"),\n]\n\n# ウィリアムズ%R(14)<-80 と同時に見る条件（2008〜2026/10 の SOMPO・他の保険株・全銘柄の検証に基づく）\nWR_CONDITIONS = [\n    Condition("up3", "陽線3日連続", "+", "全銘柄で+6.0ポイント（前半+5.8・後半+5.9）。SOMPOは該当2回のみ"),\n    Condition("dev25", "25日線乖離率 ≤ -10%", "+", "全銘柄で+3.0ポイント（前半+2.1・後半+4.2）。SOMPOは10回で-4.2ポイント"),\n    Condition("sar_flip", "パラボリックSARが買い転換（週内）", "+", "全銘柄で+2.2ポイント（前半+3.3・後半+1.2）"),\n    Condition("mkt_drop", "市場全体（全銘柄平均）が週間-3%以下", "+", "全銘柄で+6.2ポイント。ただし効果はほぼ2018年以降"),\n    Condition("above75", "終値 > 75日線", "-", "他の保険株で-7.8ポイント、全銘柄で-1.2ポイント、SOMPOで-6.5ポイント"),\n    Condition("mkt_above75", "市場全体が75日線より上", "-", "全銘柄で-3.5ポイント（後半-6.9）。SOMPOでは差なし"),\n    Condition("shallow", "週間の下げが3%未満（浅い下げ）", "?", "SOMPOで71.7%（+11.8ポイント）。全銘柄では逆向き"),\n    Condition("mkt_up", "市場全体は週間で上昇（この銘柄だけ下落）", "?", "SOMPOで72.2%（+10.5ポイント）。全銘柄では逆向き"),\n    Condition("peer_wr", "同業種の他銘柄の半数以上も%R<-80（業種ぐるみの下げ）", "?", "他の保険株で+5.0ポイント。SOMPOは+1.2ポイント"),\n    Condition("solo_wr", "同業種の他銘柄で%R<-80は25%未満（この銘柄だけの下げ）", "?", "SOMPOで70.6%（34回）。回数が少ない"),\n]\n\n\ndef _sar_bull_flip(h: np.ndarray, l: np.ndarray, af0: float = 0.02, step: float = 0.02, amax: float = 0.2) -> np.ndarray:\n    """パラボリックSAR（Wilder）が売り→買いに転換した日に True。"""\n    out = np.zeros(len(h), dtype=bool)\n    idx = np.where(np.isfinite(h) & np.isfinite(l))[0]\n    if len(idx) < 3:\n        return out\n    hh, ll = h[idx], l[idx]\n    bull, af, ep, sar = True, af0, hh[0], ll[0]\n    for i in range(1, len(idx)):\n        sar = sar + af * (ep - sar)\n        if bull:\n            sar = min(sar, ll[i - 1], ll[max(i - 2, 0)])\n            if ll[i] < sar:\n                bull, sar, ep, af = False, ep, ll[i], af0\n            elif hh[i] > ep:\n                ep, af = hh[i], min(af + step, amax)\n        else:\n            sar = max(sar, hh[i - 1], hh[max(i - 2, 0)])\n            if hh[i] > sar:\n                bull, sar, ep, af = True, ep, hh[i], af0\n                out[idx[i]] = True\n            elif ll[i] < ep:\n                ep, af = ll[i], min(af + step, amax)\n    return out\n\n\ndef _rsi(x: pd.DataFrame, n: int) -> pd.DataFrame:\n    d = x.diff()\n    w = lambda s: s.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()\n    return 100 - 100 / (1 + w(d.clip(lower=0)) / w((-d).clip(lower=0)))\n\n\ndef indicators(p: Panel, sectors: pd.Series) -> dict[str, pd.DataFrame]:\n    """日次の指標（終値基準）と、CONDITIONS の各条件の真偽。sectors: ティッカー → 業種。"""\n    C, O, H, L = p.close, p.open, p.high, p.low\n    sma = lambda n: C.rolling(n, min_periods=n).mean()\n    m25, m75, m200 = sma(25), sma(75), sma(200)\n    wret = C / C.shift(5) - 1\n    mkt = wret.mean(axis=1)  # 全銘柄の週間騰落率の単純平均\n    sec_mean = wret.T.groupby(sectors.reindex(C.columns)).transform("mean").T\n    bcast = lambda s: pd.DataFrame(np.repeat(s.to_numpy()[:, None], C.shape[1], axis=1), index=C.index, columns=C.columns)\n    down = C.diff() < 0\n    hh14, ll14 = H.rolling(14).max(), L.rolling(14).min()\n    wr14 = -100 * (hh14 - C) / (hh14 - ll14)\n    wr_on = (wr14 < WR_THRESHOLD).astype(float).where(wr14.notna())\n    sec = sectors.reindex(C.columns)\n    sec_sum = wr_on.T.groupby(sec).transform("sum").T\n    sec_cnt = wr_on.notna().astype(float).T.groupby(sec).transform("sum").T\n    peer_share = (sec_sum - wr_on.fillna(0)) / (sec_cnt - wr_on.notna()).replace(0, np.nan)  # 自分を除く同業種の割合\n    mlev = (1 + C.pct_change(fill_method=None).mean(axis=1).fillna(0)).cumprod()  # 全銘柄の日次平均騰落の累積\n    sar = pd.DataFrame({t: _sar_bull_flip(H[t].to_numpy(), L[t].to_numpy()) for t in C.columns}, index=C.index)\n    inweek = lambda df: df.astype(float).rolling(5, min_periods=1).max() > 0\n    out = {\n        "rsi2": _rsi(C, 2),\n        "wr14": wr14,\n        "peer_wr_share_pct": peer_share * 100,\n        "rsi14": _rsi(C, 14),\n        "dev25_pct": (C / m25 - 1) * 100,\n        "wret_pct": wret * 100,\n        "mkt_wret_pct": bcast(mkt * 100),\n        "sector_rel_pct": (wret - sec_mean) * 100,\n    }\n    flags = {\n        "dev25": C / m25 - 1 <= -0.10,\n        "wk10": wret <= -0.10,\n        "rsi14": out["rsi14"] < 30,\n        "mkt_drop": bcast(mkt <= -0.03),\n        "ma75_up": m75 > m75.shift(20),\n        "above200": C > m200,\n        "mkt_up": bcast(mkt > 0),\n        "sector_drop": wret - sec_mean <= -0.05,\n        "down4": down & down.shift(1, fill_value=False) & down.shift(2, fill_value=False) & down.shift(3, fill_value=False),\n        "up3": (C > O) & (C.shift(1) > O.shift(1)) & (C.shift(2) > O.shift(2)),\n        "sar_flip": inweek(sar),\n        "above75": C > m75,\n        "mkt_above75": bcast(mlev > mlev.rolling(75).mean()),\n        "shallow": wret > -0.03,\n        "peer_wr": peer_share >= 0.5,\n        "solo_wr": peer_share < 0.25,\n    }\n    out.update({f"flag_{k}": v for k, v in flags.items()})\n    return out\n\n\ndef history(p: Panel, ind: dict[str, pd.DataFrame], tickers: list[str], start: str = "2008-01-01",\n            signal: str = "rsi2", conditions: list[Condition] = CONDITIONS) -> pd.DataFrame:\n    """過去のシグナル週（判定日にシグナルあり）の一覧と、翌週の騰落率（配当込み・コスト控除前）。"""\n    sg = SIGNALS[signal]\n    mask = eligible(p)\n    ws = [w for w in weeks(p.close.index) if w.entry >= pd.Timestamp(start)]\n    ret, _ = week_returns(p, ws, cost=0.0)\n    rows = []\n    for w in ws:\n        for t in tickers:\n            v = ind[sg.key].at[w.signal, t]\n            r = ret.at[w.entry, t]\n            if not sg.on(v) or pd.isna(r) or not mask.at[w.signal, t]:\n                continue\n            row = {"ticker": t, "判定日": w.signal, "買い": w.entry, "売り": w.exit, sg.column: v, "翌週騰落率": r}\n            row.update({c.key: bool(ind[f"flag_{c.key}"].at[w.signal, t]) for c in conditions})\n            row["参考点"] = score(row, conditions)\n            rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef score(flags: dict, conditions: list[Condition] = CONDITIONS) -> int:\n    """「+」の条件数 - 「-」の条件数（「?」は数えない）。"""\n    return sum(bool(flags[c.key]) for c in conditions if c.side == "+") - sum(\n        bool(flags[c.key]) for c in conditions if c.side == "-")\n\n\ndef conditional_table(hist: pd.DataFrame, conditions: list[Condition] = CONDITIONS) -> pd.DataFrame:\n    """銘柄・条件ごとに、条件が出ていた/出ていなかったシグナル週の上昇確率。"""\n    rows = []\n    for t, g in hist.groupby("ticker"):\n        up = g["翌週騰落率"] > 0\n        for c in conditions:\n            on = g[c.key]\n            rows.append({\n                "ticker": t, "条件": c.label, "向き": c.side,\n                "あり_回数": int(on.sum()), "あり_上昇確率": up[on].mean() if on.any() else np.nan,\n                "なし_回数": int((~on).sum()), "なし_上昇確率": up[~on].mean() if (~on).any() else np.nan,\n            })\n    return pd.DataFrame(rows)\n\n\ndef latest(ind: dict[str, pd.DataFrame], tickers: list[str], day: pd.Timestamp | None = None,\n           signal: str = "rsi2", conditions: list[Condition] = CONDITIONS) -> pd.DataFrame:\n    """判定日（既定は最新日）の状態。参考点 = 「+」の条件数 - 「-」の条件数（シグナルが出ていない銘柄は空欄）。"""\n    sg = SIGNALS[signal]\n    day = day or ind[sg.key].index[-1]\n    rows = []\n    for t in tickers:\n        sig = sg.on(ind[sg.key].at[day, t])\n        row = {"ticker": t, "判定日": day.date(), "RSI2": ind["rsi2"].at[day, t], "%R14": ind["wr14"].at[day, t],\n               "シグナル": sig, "RSI14": ind["rsi14"].at[day, t], "25日線乖離%": ind["dev25_pct"].at[day, t],\n               "週間騰落%": ind["wret_pct"].at[day, t], "市場の週間騰落%": ind["mkt_wret_pct"].at[day, t],\n               "業種平均との差%": ind["sector_rel_pct"].at[day, t],\n               "同業種で%R<-80の割合%": ind["peer_wr_share_pct"].at[day, t]}\n        flags = {c.key: bool(ind[f"flag_{c.key}"].at[day, t]) for c in conditions}\n        row.update({c.label: flags[c.key] for c in conditions})\n        row["+条件の数"] = sum(flags[c.key] for c in conditions if c.side == "+")\n        row["-条件の数"] = sum(flags[c.key] for c in conditions if c.side == "-")\n        row["参考点"] = score(flags, conditions) if sig else np.nan\n        rows.append(row)\n    return pd.DataFrame(rows).set_index("ticker")\n\n\ndef trade_levels(p: Panel, ticker: str, entry: float | None = None, lookback: int = 10, rr: float = 2.0) -> dict:\n    """損切り・利確の目安（損失:利益 = 1:rr）。\n    A: 損切り＝直近 lookback 日の安値（買値の方が低ければ買値）から ATR(14)×0.1 下 / B: 損切り＝買値 − ATR(14)×2。\n    entry を省略すると最新の終値を仮の買値にする（実際は翌週初日の寄りで決まる）。"""\n    H, L, C = p.high[ticker], p.low[ticker], p.close[ticker]\n    tr = pd.concat([H - L, (H - C.shift()).abs(), (L - C.shift()).abs()], axis=1).max(axis=1)\n    atr = tr.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean().iloc[-1]\n    entry = float(entry if entry is not None else C.iloc[-1])\n    low = min(float(L.iloc[-lookback:].min()), entry)  # 寄りで直近安値を割って始まった場合は買値を基準にする\n    stop = low - 0.1 * atr\n    risk = entry - stop\n    risk_b = 2 * atr\n    return {"買値": entry, "直近安値": low, "ATR14": atr,\n            "A_損切り": stop, "A_損切り%": -risk / entry * 100, "A_利確": entry + rr * risk, "A_利確%": rr * risk / entry * 100,\n            "B_損切り": entry - risk_b, "B_損切り%": -risk_b / entry * 100,\n            "B_利確": entry + rr * risk_b, "B_利確%": rr * risk_b / entry * 100}\n\n\ndef condition_effect(p: Panel, ind: dict[str, pd.DataFrame], signal: str, conditions: list[Condition],\n                     tickers: list[str] | None = None, start: str = "2008-01-01", split: str = "2018-01-01") -> pd.DataFrame:\n    """シグナル週を条件の有無で分け、翌週の上昇確率を比べる（既定は全銘柄）。\n\n    差 = 条件あり − 条件なし（ポイント）。前半/後半は split の前後。z は2つの割合の差の検定（同じ週の銘柄は\n    独立でないため、市場全体の条件では z が大きく出すぎる点に注意）。\n    """\n    sg = SIGNALS[signal]\n    mask = eligible(p)\n    ws = [w for w in weeks(p.close.index) if w.entry >= pd.Timestamp(start)]\n    ret, _ = week_returns(p, ws, cost=0.0)\n    sig_days = [w.signal for w in ws]\n    cols = tickers or list(p.close.columns)\n    R = ret[cols].to_numpy()\n    ok = mask.loc[sig_days, cols].to_numpy() & np.isfinite(R)\n    E = (ind[sg.key].loc[sig_days, cols].to_numpy() < sg.threshold) & ok\n    UP = R > 0\n    late = np.array([w.entry >= pd.Timestamp(split) for w in ws])[:, None]\n    rows = []\n    for c in conditions:\n        X = ind[f"flag_{c.key}"].loc[sig_days, cols].fillna(False).to_numpy().astype(bool)\n        row = {"条件": c.label, "向き": c.side}\n        for lab, m in [("", np.ones_like(late)), ("前半_", ~late), ("後半_", late)]:\n            e = E & m\n            n1, n0 = int((e & X).sum()), int((e & ~X).sum())\n            k1, k0 = int((e & X & UP).sum()), int((e & ~X & UP).sum())\n            p1 = k1 / n1 if n1 else np.nan\n            p0 = k0 / n0 if n0 else np.nan\n            if lab == "":\n                pp = (k1 + k0) / max(n1 + n0, 1)\n                se = np.sqrt(pp * (1 - pp) * (1 / max(n1, 1) + 1 / max(n0, 1)))\n                row.update({"あり_回数": n1, "あり_上昇確率": p1, "なし_上昇確率": p0,\n                            "z": (p1 - p0) / se if n1 and n0 and se > 0 else np.nan})\n            row[f"{lab}差"] = (p1 - p0) * 100 if n1 and n0 else np.nan\n        rows.append(row)\n    return pd.DataFrame(rows)\n', 'backtest225/__init__.py': '"""NF日経225（1321）投資ルール検証（バックテスト）システム"""\n__version__ = "0.1.0"\n', 'backtest225/data.py': '"""価格データの取得・読み込み・クリーニング。\n\n対応ソース:\n  - Yahoo Finance（yfinance 経由の非公式API。精度・継続性は保証されない）\n  - 手持ちの日足CSV（証券会社等からダウンロードしたもの）\n\n戻り値はすべて DatetimeIndex を持ち、列 open/high/low/close/volume の DataFrame。\n価格は「分割調整済み（最新の口数基準）」にそろえる。\n"""\nfrom __future__ import annotations\n\nimport warnings\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nDEFAULT_TICKER = "1321.T"\nDATA_DIR = Path(__file__).resolve().parent.parent / "data"\nCACHE_DIR = DATA_DIR / "cache"\nBUILTIN_DIVIDENDS = {"1321.T": DATA_DIR / "dividends_1321.csv"}\n\n\n@dataclass(frozen=True)\nclass Split:\n    """受益権（株式）分割。date はその比率で価格が切り替わる最初の営業日。"""\n\n    date: pd.Timestamp\n    ratio: float  # 1口 → ratio 口\n\n\n# 1321 の既知の分割。\n# 出典: 野村アセットマネジメント「上場投資信託(ETF)の受益権分割および売買単位変更に関するお知らせ」\n#       2026年8月25日 https://nextfunds.jp/data/2026/td_260825a.pdf\n#   - 1口を100口に分割（分割基準日 2026/10/6、効力発生日 2026/10/7）\n#   - 東証の売買単位を 1口 → 10口 に変更（変更実施日 2026/10/5）\n# Yahoo Finance 上は 2026/10/5 に分割が記録され、同日から新口数基準の価格になっている。\nKNOWN_SPLITS: dict[str, list[Split]] = {\n    "1321.T": [Split(pd.Timestamp("2026-10-05"), 100.0)],\n}\n\n# 売買単位の推移（分割調整後＝現在の口数基準で表した最小売買口数）。\n# 2026/10/5 より前は 旧1口 = 新100口 が最小単位、以降は 10口。\nKNOWN_LOT_SCHEDULE: dict[str, list[tuple[pd.Timestamp, int]]] = {\n    "1321.T": [(pd.Timestamp("1900-01-01"), 100), (pd.Timestamp("2026-10-05"), 10)],\n}\n\n# CSVの列名ゆれを吸収する対応表（小文字化・空白除去後に照合）\n_COLUMN_ALIASES = {\n    "date": ["date", "日付", "年月日", "datetime", "time"],\n    "open": ["open", "始値"],\n    "high": ["high", "高値"],\n    "low": ["low", "安値"],\n    "close": ["close", "終値", "close price"],\n    "volume": ["volume", "出来高", "vol"],\n}\n\n\ndef _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:\n    rename = {}\n    for col in df.columns:\n        key = str(col).strip().lower()\n        for std, aliases in _COLUMN_ALIASES.items():\n            if key in aliases and std not in rename.values():\n                rename[col] = std\n                break\n    return df.rename(columns=rename)\n\n\ndef load_csv(path: str | Path, encoding: str | None = None) -> pd.DataFrame:\n    """日足CSVを読み込む。UTF-8 / Shift_JIS(cp932) を自動判定。"""\n    path = Path(path)\n    encodings = [encoding] if encoding else ["utf-8-sig", "cp932"]\n    last_err: Exception | None = None\n    for enc in encodings:\n        try:\n            raw = pd.read_csv(path, encoding=enc)\n            break\n        except UnicodeDecodeError as e:  # 次のエンコーディングを試す\n            last_err = e\n    else:\n        raise ValueError(f"CSVの文字コードを判定できません: {path}") from last_err\n\n    df = _normalize_columns(raw)\n    missing = {"date", "close"} - set(df.columns)\n    if missing:\n        raise ValueError(\n            f"CSVに必要な列がありません: {sorted(missing)}。"\n            f"検出した列: {list(raw.columns)}（日付・終値は必須。始値/高値/安値/出来高は推奨）"\n        )\n    df["date"] = pd.to_datetime(df["date"])\n    df = df.set_index("date").sort_index()\n    for c in ["open", "high", "low", "close", "volume"]:\n        if c in df.columns:\n            df[c] = pd.to_numeric(\n                df[c].astype(str).str.replace(",", "", regex=False), errors="coerce"\n            )\n    return df[[c for c in ["open", "high", "low", "close", "volume"] if c in df.columns]]\n\n\ndef fetch_yahoo(ticker: str = DEFAULT_TICKER, use_cache: bool = True, refresh: bool = False) -> pd.DataFrame:\n    """Yahoo Finance から日足を取得（非調整価格、分割のみ調整済み）。キャッシュあり。"""\n    CACHE_DIR.mkdir(parents=True, exist_ok=True)\n    cache = CACHE_DIR / f"{ticker.replace(\'.\', \'_\')}.csv"\n    if use_cache and cache.exists() and not refresh:\n        df = pd.read_csv(cache, index_col=0, parse_dates=True)\n        return df\n\n    try:\n        import yfinance as yf\n    except ImportError as e:  # pragma: no cover\n        raise ImportError("yfinance が未インストールです: pip install yfinance") from e\n\n    raw = yf.download(ticker, period="max", auto_adjust=False, progress=False, actions=False)\n    if raw is None or raw.empty:\n        raise RuntimeError(f"Yahoo Finance からデータを取得できませんでした: {ticker}")\n    if isinstance(raw.columns, pd.MultiIndex):\n        raw = raw.droplevel(1, axis=1)\n    df = raw.rename(columns=str.lower)[["open", "high", "low", "close", "volume"]]\n    df.index = pd.to_datetime(df.index).tz_localize(None)\n    df.index.name = "date"\n    if use_cache:\n        df.to_csv(cache)\n    return df\n\n\ndef fetch_yahoo_dividends(ticker: str, prices: pd.DataFrame) -> pd.Series:\n    """Yahoo Finance の分配金（1口あたり、現在の口数基準へ換算）。\n\n    Yahoo の分配金データは分割調整が不統一なことがある（1321では2025年以前が旧口数基準のまま）。\n    そこで「分配金 / その日の終値 > 50%」のものは旧口数基準とみなし、既知の分割比率で割る。\n    非公式データのため、正確な検証には運用会社公表値のCSVを使うこと。\n    """\n    import yfinance as yf\n\n    div = yf.Ticker(ticker).dividends\n    if div is None or div.empty:\n        return pd.Series(dtype=float)\n    div.index = pd.to_datetime(div.index).tz_localize(None).normalize()\n    return normalize_dividends(div, prices, KNOWN_SPLITS.get(ticker, []))\n\n\ndef normalize_dividends(div: pd.Series, prices: pd.DataFrame, splits: list[Split]) -> pd.Series:\n    div = div.astype(float).copy()\n    closes = prices["close"].reindex(div.index, method="ffill")\n    for sp in splits:\n        mask = (div.index < sp.date) & (div / closes > 0.5)\n        div[mask] = div[mask] / sp.ratio\n    return div[div > 0]\n\n\ndef load_dividends_csv(path: str | Path) -> pd.Series:\n    """分配金CSV（列: date, amount）。date は決算日(基準日)、amount は価格と同じ口数基準の1口あたり金額。"""\n    df = pd.read_csv(path, comment="#")\n    df = df.rename(columns={c: str(c).strip().lower() for c in df.columns})\n    df = df.rename(columns={"日付": "date", "権利落ち日": "date", "分配金": "amount"})\n    s = pd.Series(pd.to_numeric(df["amount"], errors="coerce").values, index=pd.to_datetime(df["date"]))\n    return s.dropna().sort_index()\n\n\nT_PLUS_2_START = pd.Timestamp("2019-07-16")  # 受渡しがT+2になった約定日（JPX）\n\n\ndef to_ex_dates(div: pd.Series, index: pd.DatetimeIndex) -> pd.Series:\n    """基準日(決算日)の分配金を、権利落ち日（価格が分配金分下がる営業日）に付け替える。\n\n    権利付最終日 = 受渡日が基準日以前になる最後の営業日（T+3、2019/7/16約定分からT+2）。\n    権利落ち日 = その翌営業日。営業日は価格データの日付で近似する。\n    データ期間外の分配金は除外する。\n    """\n    out = {}\n    pos = np.arange(len(index))\n    for rec, amt in div.items():\n        rec = pd.Timestamp(rec).normalize()\n        ok = []\n        for j in pos:\n            s = 2 if index[j] >= T_PLUS_2_START else 3\n            if j + s < len(index) and index[j + s] <= rec:\n                ok.append(j)\n            elif j + s < len(index):\n                break\n        if not ok or ok[-1] + 1 >= len(index):\n            continue\n        ex = index[ok[-1] + 1]\n        out[ex] = out.get(ex, 0.0) + float(amt)\n    return pd.Series(out, dtype=float).sort_index()\n\n\ndef adjust_splits(df: pd.DataFrame, splits: list[Split]) -> tuple[pd.DataFrame, list[str]]:\n    """未調整の価格（分割前後で水準が飛んでいる）を検出した場合のみ、分割前の価格を調整する。\n\n    すでに調整済みのデータ（Yahoo等）には何もしない。\n    """\n    notes: list[str] = []\n    df = df.copy()\n    for sp in splits:\n        # 分割日前後それぞれ5営業日の中央値で比べる（1日だけの異常値に引きずられないように）\n        before = df.loc[df.index < sp.date, "close"].tail(5)\n        after = df.loc[df.index >= sp.date, "close"].head(5)\n        if before.empty or after.empty:\n            continue\n        jump = before.median() / after.median()\n        if jump > sp.ratio / 2:  # 分割前後で 1/ratio 近い急落 → 未調整と判断\n            cols = [c for c in ["open", "high", "low", "close"] if c in df.columns]\n            df.loc[df.index < sp.date, cols] /= sp.ratio\n            if "volume" in df.columns:\n                df.loc[df.index < sp.date, "volume"] *= sp.ratio\n            notes.append(f"{sp.date.date()} の 1:{sp.ratio:g} 分割を検出し、それ以前の価格を調整しました")\n    return df, notes\n\n\ndef clean(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:\n    """欠損・異常値の補正。補正内容は notes として返し、レポートに明記する。"""\n    notes: list[str] = []\n    df = df.copy()\n    df = df[~df.index.duplicated(keep="last")].sort_index()\n    df = df[df["close"].notna() & (df["close"] > 0)]\n\n    # 前後の営業日から桁違い（約1/10・1/100・10倍・100倍）にずれた日を補正する（配信元の分割処理の誤り対策）\n    ref = df["close"].rolling(11, center=True, min_periods=3).median()\n    ratio = df["close"] / ref\n    mag = np.round(np.log10(ratio))\n    bad = (mag != 0) & ((np.log10(ratio) - mag).abs() < 0.15)\n    if bad.any():\n        k = 10.0 ** (-mag[bad])\n        for col in ["open", "high", "low", "close"]:\n            if col in df.columns:\n                df.loc[bad, col] = df.loc[bad, col] * k\n        if "volume" in df.columns:\n            df.loc[bad, "volume"] = df.loc[bad, "volume"] / k\n        notes.append("前後と桁が違う価格の日を補正しました: " + ", ".join(f"{d.date()}(×{v:g})" for d, v in k.items()))\n\n    for c in ["open", "high", "low"]:\n        if c not in df.columns:\n            df[c] = np.nan\n    if "volume" not in df.columns:\n        df["volume"] = np.nan\n\n    bad_open = df["open"].isna() | (df["open"] <= 0)\n    if bad_open.any():\n        n = int(bad_open.sum())\n        last = df.index[bad_open][-1].date()\n        df.loc[bad_open, "open"] = df["close"].shift(1)[bad_open]\n        df["open"] = df["open"].fillna(df["close"])\n        notes.append(\n            f"始値が欠損/0の日が {n} 日（最終 {last}）あり、前日終値で代用しました（約定価格は近似）"\n        )\n\n    for c in ["high", "low"]:\n        bad = df[c].isna() | (df[c] <= 0)\n        if bad.any():\n            notes.append(f"{c} が欠損/0の日が {int(bad.sum())} 日あり、始値・終値から補完しました")\n            df.loc[bad, c] = np.nan\n    df["high"] = df[["high", "open", "close"]].max(axis=1)\n    df["low"] = df[["low", "open", "close"]].min(axis=1)\n\n    ret = df["close"].pct_change().abs()\n    jumps = ret[ret > 0.2]\n    if not jumps.empty:\n        notes.append(\n            "前日比±20%超の値動きがあります（データ異常/未処理の分割の可能性）: "\n            + ", ".join(str(d.date()) for d in jumps.index[:5])\n        )\n    return df[["open", "high", "low", "close", "volume"]], notes\n\n\ndef load_prices(\n    source: str = "yahoo",\n    ticker: str = DEFAULT_TICKER,\n    csv_path: str | None = None,\n    start: str | None = None,\n    end: str | None = None,\n    refresh: bool = False,\n) -> tuple[pd.DataFrame, list[str]]:\n    """データソースに応じて価格を読み込み、分割調整・クリーニングして返す。"""\n    notes: list[str] = []\n    if source == "csv":\n        if not csv_path:\n            raise ValueError("source=csv の場合は csv パスを指定してください")\n        df = load_csv(csv_path)\n        notes.append(f"データ: CSV {csv_path}")\n    elif source == "yahoo":\n        df = fetch_yahoo(ticker, refresh=refresh)\n        notes.append(f"データ: Yahoo Finance {ticker}（非公式API。値は一次情報で要確認）")\n    else:\n        raise ValueError(f"未知のデータソース: {source}")\n\n    df, n1 = adjust_splits(df, KNOWN_SPLITS.get(ticker, []))\n    df, n2 = clean(df)\n    notes += n1 + n2\n    if start:\n        df = df[df.index >= pd.Timestamp(start)]\n    if end:\n        df = df[df.index <= pd.Timestamp(end)]\n    if len(df) < 2:\n        raise ValueError("指定期間のデータが2日未満です")\n    return df, notes\n\n\ndef lot_size_series(index: pd.DatetimeIndex, ticker: str, lot_size: int | str | None) -> pd.Series:\n    """各営業日の最小売買口数。lot_size=\'auto\' なら既知のスケジュール、数値なら固定。"""\n    if lot_size in (None, "auto"):\n        sched = KNOWN_LOT_SCHEDULE.get(ticker)\n        if sched is None:\n            warnings.warn(f"{ticker} の売買単位が未登録のため 1 口単位として扱います")\n            return pd.Series(1, index=index)\n        s = pd.Series(np.nan, index=index)\n        for d, lot in sched:\n            s[index >= d] = lot\n        return s.astype(int)\n    return pd.Series(int(lot_size), index=index)\n'}
UNIVERSE_CSV = 'list_date,code,name,market,sector,size\n20260930,1332,ニッスイ,プライム（内国株式）,水産・農林業,TOPIX Mid400\n20260930,1333,Ｕｍｉｏｓ,プライム（内国株式）,水産・農林業,TOPIX Mid400\n20260930,1414,ショーボンドホールディングス,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1417,ミライト・ワン,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1605,ＩＮＰＥＸ,プライム（内国株式）,鉱業,TOPIX Mid400\n20260930,1719,安藤・間,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1721,コムシスホールディングス,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1801,大成建設,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1802,大林組,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1803,清水建設,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1808,長谷工コーポレーション,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1812,鹿島建設,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1820,西松建設,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1860,戸田建設,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1878,大東建託,プライム（内国株式）,不動産業,TOPIX Mid400\n20260930,1893,五洋建設,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1911,住友林業,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1925,大和ハウス工業,プライム（内国株式）,建設業,TOPIX Large70\n20260930,1928,積水ハウス,プライム（内国株式）,建設業,TOPIX Large70\n20260930,1942,関電工,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1944,きんでん,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1951,エクシオグループ,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1959,クラフティア,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1963,日揮ホールディングス,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,1969,高砂熱学工業,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,2002,日清製粉グループ本社,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2127,日本Ｍ＆Ａセンターホールディングス,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,2181,パーソルホールディングス,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,2201,森永製菓,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2206,江崎グリコ,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2212,山崎製パン,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2222,寿スピリッツ,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2229,カルビー,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2264,森永乳業,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2267,ヤクルト本社,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2269,明治ホールディングス,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2282,日本ハム,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2327,日鉄ソリューションズ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,2331,ＡＬＳＯＫ,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,2371,カカクコム,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,2413,エムスリー,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,2432,ディー・エヌ・エー,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,2433,博報堂ＤＹホールディングス,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,2501,サッポロビール,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2502,アサヒグループホールディングス,プライム（内国株式）,食料品,TOPIX Large70\n20260930,2503,キリンホールディングス,プライム（内国株式）,食料品,TOPIX Large70\n20260930,2531,宝ホールディングス,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2579,コカ・コーラ\u3000ボトラーズジャパンホールディングス,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2587,サントリービバレッジ＆フード,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2593,伊藤園,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2607,不二製油,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2670,エービーシー・マート,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,2768,双日,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,2784,アルフレッサ\u3000ホールディングス,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,2801,キッコーマン,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2802,味の素,プライム（内国株式）,食料品,TOPIX Large70\n20260930,2809,キユーピー,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2810,ハウス食品グループ本社,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2811,カゴメ,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,285A,キオクシアホールディングス,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,2871,ニチレイ,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2875,東洋水産,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2897,日清食品ホールディングス,プライム（内国株式）,食料品,TOPIX Mid400\n20260930,2914,日本たばこ産業,プライム（内国株式）,食料品,TOPIX Core30\n20260930,3003,ヒューリック,プライム（内国株式）,不動産業,TOPIX Mid400\n20260930,3038,神戸物産,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,3048,ビックカメラ,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3064,ＭｏｎｏｔａＲＯ,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3086,Ｊ．フロント\u3000リテイリング,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3088,マツキヨココカラ＆カンパニー,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3092,ＺＯＺＯ,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3099,三越伊勢丹ホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3105,日清紡ホールディングス,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,3107,ダイワボウホールディングス,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,3116,トヨタ紡織,プライム（内国株式）,輸送用機器,TOPIX Mid400\n20260930,3132,マクニカホールディングス,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,3197,すかいらーくホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3231,野村不動産ホールディングス,プライム（内国株式）,不動産業,TOPIX Mid400\n20260930,3288,オープンハウスグループ,プライム（内国株式）,不動産業,TOPIX Mid400\n20260930,3289,東急不動産ホールディングス,プライム（内国株式）,不動産業,TOPIX Mid400\n20260930,3291,飯田グループホールディングス,プライム（内国株式）,不動産業,TOPIX Mid400\n20260930,3349,コスモス薬品,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3360,シップヘルスケアホールディングス,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,3382,セブン＆アイ・ホールディングス,プライム（内国株式）,小売業,TOPIX Core30\n20260930,3391,ツルハホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3397,トリドールホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3401,帝人,プライム（内国株式）,繊維製品,TOPIX Mid400\n20260930,3402,東レ,プライム（内国株式）,繊維製品,TOPIX Mid400\n20260930,3405,クラレ,プライム（内国株式）,化学,TOPIX Mid400\n20260930,3407,旭化成,プライム（内国株式）,化学,TOPIX Large70\n20260930,3436,ＳＵＭＣＯ,プライム（内国株式）,金属製品,TOPIX Mid400\n20260930,3549,クスリのアオキホールディングス,スタンダード（内国株式）,小売業,TOPIX Mid400\n20260930,3563,ＦＯＯＤ\u3000＆\u3000ＬＩＦＥ\u3000ＣＯＭＰＡＮＩＥＳ,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,3591,ワコールホールディングス,プライム（内国株式）,繊維製品,TOPIX Mid400\n20260930,3626,ＴＩＳＩ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,3635,コーエーテクモホールディングス,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,3659,ネクソン,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,3697,ＳＨＩＦＴ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,3769,ＧＭＯペイメントゲートウェイ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,3774,インターネットイニシアティブ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,3861,王子ホールディングス,プライム（内国株式）,パルプ・紙,TOPIX Mid400\n20260930,3923,ラクス,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,3941,レンゴー,プライム（内国株式）,パルプ・紙,TOPIX Mid400\n20260930,3994,マネーフォワード,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,4004,レゾナック・ホールディングス,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4005,住友化学,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4021,日産化学,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4042,東ソー,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4043,トクヤマ,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4045,東亞合成,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4061,デンカ,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4062,イビデン,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,4063,信越化学工業,プライム（内国株式）,化学,TOPIX Core30\n20260930,4091,日本酸素ホールディングス,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4114,日本触媒,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4118,カネカ,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4151,協和キリン,プライム（内国株式）,医薬品,TOPIX Mid400\n20260930,417A,ブルーゾーンホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,4182,三菱瓦斯化学,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4183,三井化学,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4186,東京応化工業,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4188,三菱ケミカルグループ,プライム（内国株式）,化学,TOPIX Large70\n20260930,4194,ビジョナル,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,4202,ダイセル,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4203,住友ベークライト,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4204,積水化学工業,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4205,日本ゼオン,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4206,アイカ工業,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4208,ＵＢＥ,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4272,日本化薬,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4307,野村総合研究所,プライム（内国株式）,情報・通信業,TOPIX Large70\n20260930,4324,電通グループ,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,4385,メルカリ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,4401,ＡＤＥＫＡ,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4403,日油,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4452,花王,プライム（内国株式）,化学,TOPIX Large70\n20260930,4502,武田薬品工業,プライム（内国株式）,医薬品,TOPIX Core30\n20260930,4503,アステラス製薬,プライム（内国株式）,医薬品,TOPIX Large70\n20260930,4506,住友ファーマ,プライム（内国株式）,医薬品,TOPIX Mid400\n20260930,4507,塩野義製薬,プライム（内国株式）,医薬品,TOPIX Large70\n20260930,4516,日本新薬,プライム（内国株式）,医薬品,TOPIX Mid400\n20260930,4519,中外製薬,プライム（内国株式）,医薬品,TOPIX Large70\n20260930,4523,エーザイ,プライム（内国株式）,医薬品,TOPIX Mid400\n20260930,4527,ロート製薬,プライム（内国株式）,医薬品,TOPIX Mid400\n20260930,4528,小野薬品工業,プライム（内国株式）,医薬品,TOPIX Mid400\n20260930,4536,参天製薬,プライム（内国株式）,医薬品,TOPIX Mid400\n20260930,4540,ツムラ,プライム（内国株式）,医薬品,TOPIX Mid400\n20260930,4543,テルモ,プライム（内国株式）,精密機器,TOPIX Large70\n20260930,4544,Ｈ．Ｕ．グループホールディングス,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,4568,第一三共,プライム（内国株式）,医薬品,TOPIX Core30\n20260930,4578,大塚ホールディングス,プライム（内国株式）,医薬品,TOPIX Large70\n20260930,4587,ペプチドリーム,プライム（内国株式）,医薬品,TOPIX Mid400\n20260930,4612,日本ペイントホールディングス,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4613,関西ペイント,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4626,太陽ホールディングス,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4631,ＤＩＣ,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4661,オリエンタルランド,プライム（内国株式）,サービス業,TOPIX Large70\n20260930,4665,ダスキン,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,4666,パーク２４,プライム（内国株式）,不動産業,TOPIX Mid400\n20260930,4676,フジ・メディア・ホールディングス,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,4680,ラウンドワン,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,4681,リゾートトラスト,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,4684,オービック,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,4689,ＬＩＮＥヤフー,プライム（内国株式）,情報・通信業,TOPIX Large70\n20260930,4704,トレンドマイクロ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,4716,日本オラクル,スタンダード（内国株式）,情報・通信業,TOPIX Mid400\n20260930,4732,ユー・エス・エス,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,4733,オービックビジネスコンサルタント,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,4751,サイバーエージェント,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,4755,楽天グループ,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,4768,大塚商会,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,4887,サワイグループホールディングス,プライム（内国株式）,医薬品,TOPIX Mid400\n20260930,4901,富士フイルムホールディングス,プライム（内国株式）,化学,TOPIX Large70\n20260930,4902,コニカミノルタ,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,4911,資生堂,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4912,ライオン,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4922,コーセーホールディングス,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4927,ポーラ・オルビスホールディングス,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4967,小林製薬,プライム（内国株式）,化学,TOPIX Mid400\n20260930,4980,デクセリアルズ,プライム（内国株式）,化学,TOPIX Mid400\n20260930,5016,ＪＸ金属,プライム（内国株式）,非鉄金属,TOPIX Mid400\n20260930,5019,出光興産,プライム（内国株式）,石油・石炭製品,TOPIX Mid400\n20260930,5020,ＥＮＥＯＳホールディングス,プライム（内国株式）,石油・石炭製品,TOPIX Large70\n20260930,5021,コスモエネルギーホールディングス,プライム（内国株式）,石油・石炭製品,TOPIX Mid400\n20260930,5076,インフロニア・ホールディングス,プライム（内国株式）,建設業,TOPIX Mid400\n20260930,5101,横浜ゴム,プライム（内国株式）,ゴム製品,TOPIX Mid400\n20260930,5105,ＴＯＹＯ\u3000ＴＩＲＥ,プライム（内国株式）,ゴム製品,TOPIX Mid400\n20260930,5108,ブリヂストン,プライム（内国株式）,ゴム製品,TOPIX Large70\n20260930,5110,住友ゴム工業,プライム（内国株式）,ゴム製品,TOPIX Mid400\n20260930,5201,ＡＧＣ,プライム（内国株式）,ガラス・土石製品,TOPIX Mid400\n20260930,5214,日本電気硝子,プライム（内国株式）,ガラス・土石製品,TOPIX Mid400\n20260930,5232,住友大阪セメント,プライム（内国株式）,ガラス・土石製品,TOPIX Mid400\n20260930,5233,太平洋セメント,プライム（内国株式）,ガラス・土石製品,TOPIX Mid400\n20260930,5301,東海カーボン,プライム（内国株式）,ガラス・土石製品,TOPIX Mid400\n20260930,5332,ＴＯＴＯ,プライム（内国株式）,ガラス・土石製品,TOPIX Mid400\n20260930,5333,ＮＧＫ,プライム（内国株式）,ガラス・土石製品,TOPIX Mid400\n20260930,5334,日本特殊陶業,プライム（内国株式）,ガラス・土石製品,TOPIX Mid400\n20260930,5344,ＭＡＲＵＷＡ,プライム（内国株式）,ガラス・土石製品,TOPIX Mid400\n20260930,5393,ニチアス,プライム（内国株式）,ガラス・土石製品,TOPIX Mid400\n20260930,5401,日本製鉄,プライム（内国株式）,鉄鋼,TOPIX Large70\n20260930,5406,神戸製鋼所,プライム（内国株式）,鉄鋼,TOPIX Mid400\n20260930,5411,ＪＦＥホールディングス,プライム（内国株式）,鉄鋼,TOPIX Mid400\n20260930,5444,大和工業,プライム（内国株式）,鉄鋼,TOPIX Mid400\n20260930,5463,丸一鋼管,プライム（内国株式）,鉄鋼,TOPIX Mid400\n20260930,5471,大同特殊鋼,プライム（内国株式）,鉄鋼,TOPIX Mid400\n20260930,5631,日本製鋼所,プライム（内国株式）,機械,TOPIX Mid400\n20260930,5706,三井金属,プライム（内国株式）,非鉄金属,TOPIX Mid400\n20260930,5711,三菱マテリアル,プライム（内国株式）,非鉄金属,TOPIX Mid400\n20260930,5713,住友金属鉱山,プライム（内国株式）,非鉄金属,TOPIX Mid400\n20260930,5714,ＤＯＷＡホールディングス,プライム（内国株式）,非鉄金属,TOPIX Mid400\n20260930,5801,古河電気工業,プライム（内国株式）,非鉄金属,TOPIX Mid400\n20260930,5802,住友電気工業,プライム（内国株式）,非鉄金属,TOPIX Large70\n20260930,5803,フジクラ,プライム（内国株式）,非鉄金属,TOPIX Large70\n20260930,5830,いよぎんホールディングス,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,5831,しずおかフィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,5832,ちゅうぎんフィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,5838,楽天銀行,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,5844,京都フィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,5901,東洋製罐グループホールディングス,プライム（内国株式）,金属製品,TOPIX Mid400\n20260930,5929,三和ホールディングス,プライム（内国株式）,金属製品,TOPIX Mid400\n20260930,5938,ＬＩＸＩＬ,プライム（内国株式）,金属製品,TOPIX Mid400\n20260930,5947,リンナイ,プライム（内国株式）,金属製品,TOPIX Mid400\n20260930,5991,日本発條,プライム（内国株式）,金属製品,TOPIX Mid400\n20260930,6005,三浦工業,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6098,リクルートホールディングス,プライム（内国株式）,サービス業,TOPIX Core30\n20260930,6113,アマダ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6134,ＦＵＪＩ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6136,オーエスジー,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6141,ＤＭＧ森精機,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6146,ディスコ,プライム（内国株式）,機械,TOPIX Large70\n20260930,6178,日本郵政,プライム（内国株式）,サービス業,TOPIX Large70\n20260930,6268,ナブテスコ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6269,三井海洋開発,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6273,ＳＭＣ,プライム（内国株式）,機械,TOPIX Large70\n20260930,6301,小松製作所,プライム（内国株式）,機械,TOPIX Large70\n20260930,6302,住友重機械工業,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6305,日立建機,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6323,ローツェ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6326,クボタ,プライム（内国株式）,機械,TOPIX Large70\n20260930,6361,荏原製作所,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6367,ダイキン工業,プライム（内国株式）,機械,TOPIX Core30\n20260930,6368,オルガノ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6370,栗田工業,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6383,ダイフク,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6417,ＳＡＮＫＹＯ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6436,アマノ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6448,ブラザー工業,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6457,グローリー,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6460,セガサミーホールディングス,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6465,ホシザキ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,646A,クラサスケミカル,スタンダード（内国株式）,化学,TOPIX Mid400\n20260930,6471,日本精工,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6472,ＮＴＮ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6473,ジェイテクト,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6479,ミネベアミツミ,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6481,ＴＨＫ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6501,日立製作所,プライム（内国株式）,電気機器,TOPIX Core30\n20260930,6503,三菱電機,プライム（内国株式）,電気機器,TOPIX Core30\n20260930,6504,富士電機,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6506,安川電機,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6525,ＫＯＫＵＳＡＩ\u3000ＥＬＥＣＴＲＩＣ,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6526,ソシオネクスト,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6532,ベイカレント,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,6544,ジャパンエレベーターサービスホールディングス,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,6586,マキタ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,6592,マブチモーター,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6632,ＪＶＣケンウッド,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6645,オムロン,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6674,ジーエス・ユアサ\u3000コーポレーション,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6701,日本電気,プライム（内国株式）,電気機器,TOPIX Large70\n20260930,6702,富士通,プライム（内国株式）,電気機器,TOPIX Large70\n20260930,6707,サンケン電気,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6723,ルネサスエレクトロニクス,プライム（内国株式）,電気機器,TOPIX Large70\n20260930,6724,セイコーエプソン,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6728,アルバック,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6752,パナソニック\u3000ホールディングス,プライム（内国株式）,電気機器,TOPIX Large70\n20260930,6753,シャープ,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6754,アンリツ,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6758,ソニーグループ,プライム（内国株式）,電気機器,TOPIX Core30\n20260930,6762,ＴＤＫ,プライム（内国株式）,電気機器,TOPIX Large70\n20260930,6770,アルプスアルパイン,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6806,ヒロセ電機,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6841,横河電機,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6845,アズビル,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6849,日本光電工業,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6856,堀場製作所,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6857,アドバンテスト,プライム（内国株式）,電気機器,TOPIX Core30\n20260930,6861,キーエンス,プライム（内国株式）,電気機器,TOPIX Core30\n20260930,6869,シスメックス,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6902,デンソー,プライム（内国株式）,輸送用機器,TOPIX Large70\n20260930,6920,レーザーテック,プライム（内国株式）,電気機器,TOPIX Large70\n20260930,6923,スタンレー電気,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6925,ウシオ電機,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6951,日本電子,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6952,カシオ計算機,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6954,ファナック,プライム（内国株式）,電気機器,TOPIX Large70\n20260930,6963,ローム,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6965,浜松ホトニクス,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6971,京セラ,プライム（内国株式）,電気機器,TOPIX Large70\n20260930,6976,太陽誘電,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,6981,村田製作所,プライム（内国株式）,電気機器,TOPIX Large70\n20260930,6988,日東電工,プライム（内国株式）,化学,TOPIX Mid400\n20260930,7003,三井Ｅ＆Ｓ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,7011,三菱重工業,プライム（内国株式）,機械,TOPIX Core30\n20260930,7012,川崎重工業,プライム（内国株式）,輸送用機器,TOPIX Mid400\n20260930,7013,ＩＨＩ,プライム（内国株式）,機械,TOPIX Mid400\n20260930,7164,全国保証,プライム（内国株式）,その他金融業,TOPIX Mid400\n20260930,7167,めぶきフィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,7180,九州フィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,7181,かんぽ生命保険,プライム（内国株式）,保険業,TOPIX Mid400\n20260930,7182,ゆうちょ銀行,プライム（内国株式）,銀行業,TOPIX Large70\n20260930,7186,横浜フィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,7189,西日本フィナンシャルホールディングス,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,7201,日産自動車,プライム（内国株式）,輸送用機器,TOPIX Mid400\n20260930,7202,いすゞ自動車,プライム（内国株式）,輸送用機器,TOPIX Mid400\n20260930,7203,トヨタ自動車,プライム（内国株式）,輸送用機器,TOPIX Core30\n20260930,7211,三菱自動車工業,プライム（内国株式）,輸送用機器,TOPIX Mid400\n20260930,7259,アイシン,プライム（内国株式）,輸送用機器,TOPIX Mid400\n20260930,7261,マツダ,プライム（内国株式）,輸送用機器,TOPIX Mid400\n20260930,7267,本田技研工業,プライム（内国株式）,輸送用機器,TOPIX Core30\n20260930,7269,スズキ,プライム（内国株式）,輸送用機器,TOPIX Large70\n20260930,7270,ＳＵＢＡＲＵ,プライム（内国株式）,輸送用機器,TOPIX Large70\n20260930,7272,ヤマハ発動機,プライム（内国株式）,輸送用機器,TOPIX Mid400\n20260930,7276,小糸製作所,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,7282,豊田合成,プライム（内国株式）,輸送用機器,TOPIX Mid400\n20260930,7309,シマノ,プライム（内国株式）,輸送用機器,TOPIX Large70\n20260930,7313,テイ・エス\u3000テック,プライム（内国株式）,輸送用機器,TOPIX Mid400\n20260930,7337,ひろぎんホールディングス,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,7453,良品計画,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,7458,第一興商,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,7459,メディパルホールディングス,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,7476,アズワン,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,7532,パン・パシフィック・インターナショナルホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,7550,ゼンショーホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,7616,コロワイド,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,7649,スギホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,7701,島津製作所,プライム（内国株式）,精密機器,TOPIX Mid400\n20260930,7729,東京精密,プライム（内国株式）,精密機器,TOPIX Mid400\n20260930,7731,ニコン,プライム（内国株式）,精密機器,TOPIX Mid400\n20260930,7733,オリンパス,プライム（内国株式）,精密機器,TOPIX Large70\n20260930,7735,ＳＣＲＥＥＮホールディングス,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,7741,ＨＯＹＡ,プライム（内国株式）,精密機器,TOPIX Core30\n20260930,7747,朝日インテック,プライム（内国株式）,精密機器,TOPIX Mid400\n20260930,7751,キヤノン,プライム（内国株式）,電気機器,TOPIX Large70\n20260930,7752,リコー,プライム（内国株式）,電気機器,TOPIX Mid400\n20260930,7832,バンダイナムコホールディングス,プライム（内国株式）,その他製品,TOPIX Large70\n20260930,7846,パイロットコーポレーション,プライム（内国株式）,その他製品,TOPIX Mid400\n20260930,7867,タカラトミー,プライム（内国株式）,その他製品,TOPIX Mid400\n20260930,7911,ＴＯＰＰＡＮホールディングス,プライム（内国株式）,その他製品,TOPIX Mid400\n20260930,7912,大日本印刷,プライム（内国株式）,その他製品,TOPIX Mid400\n20260930,7936,アシックス,プライム（内国株式）,その他製品,TOPIX Large70\n20260930,7951,ヤマハ,プライム（内国株式）,その他製品,TOPIX Mid400\n20260930,7956,ピジョン,プライム（内国株式）,その他製品,TOPIX Mid400\n20260930,7974,任天堂,プライム（内国株式）,その他製品,TOPIX Core30\n20260930,7984,コクヨ,プライム（内国株式）,その他製品,TOPIX Mid400\n20260930,7988,ニフコ,プライム（内国株式）,化学,TOPIX Mid400\n20260930,8001,伊藤忠商事,プライム（内国株式）,卸売業,TOPIX Core30\n20260930,8002,丸紅,プライム（内国株式）,卸売業,TOPIX Large70\n20260930,8012,長瀬産業,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,8015,豊田通商,プライム（内国株式）,卸売業,TOPIX Large70\n20260930,8020,兼松,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,8031,三井物産,プライム（内国株式）,卸売業,TOPIX Core30\n20260930,8035,東京エレクトロン,プライム（内国株式）,電気機器,TOPIX Core30\n20260930,8053,住友商事,プライム（内国株式）,卸売業,TOPIX Large70\n20260930,8056,ＢＩＰＲＯＧＹ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,8058,三菱商事,プライム（内国株式）,卸売業,TOPIX Core30\n20260930,8060,キヤノンマーケティングジャパン,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,8086,ニプロ,プライム（内国株式）,精密機器,TOPIX Mid400\n20260930,8088,岩谷産業,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,8111,ゴールドウイン,プライム（内国株式）,繊維製品,TOPIX Mid400\n20260930,8113,ユニ・チャーム,プライム（内国株式）,化学,TOPIX Large70\n20260930,8129,東邦ホールディングス,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,8136,サンリオ,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,8174,日本瓦斯,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,8227,しまむら,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,8233,高島屋,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,8242,エイチ・ツー・オー\u3000リテイリング,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,8252,丸井グループ,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,8253,クレディセゾン,プライム（内国株式）,その他金融業,TOPIX Mid400\n20260930,8267,イオン,プライム（内国株式）,小売業,TOPIX Large70\n20260930,8282,ケーズホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,8303,ＳＢＩ新生銀行,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,8304,あおぞら銀行,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,8306,三菱ＵＦＪフィナンシャル・グループ,プライム（内国株式）,銀行業,TOPIX Core30\n20260930,8308,りそなホールディングス,プライム（内国株式）,銀行業,TOPIX Large70\n20260930,8309,三井住友トラストグループ,プライム（内国株式）,銀行業,TOPIX Large70\n20260930,8316,三井住友フィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Core30\n20260930,8331,千葉銀行,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,8334,群馬銀行,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,8341,七十七銀行,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,8354,ふくおかフィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,8359,八十二長野銀行,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,8377,ほくほくフィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,8410,セブン銀行,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,8411,みずほフィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Core30\n20260930,8418,山口フィナンシャルグループ,プライム（内国株式）,銀行業,TOPIX Mid400\n20260930,8424,芙蓉総合リース,プライム（内国株式）,その他金融業,TOPIX Mid400\n20260930,8439,東京センチュリー,プライム（内国株式）,その他金融業,TOPIX Mid400\n20260930,8473,ＳＢＩホールディングス,プライム（内国株式）,証券、商品先物取引業,TOPIX Mid400\n20260930,8570,イオンフィナンシャルサービス,プライム（内国株式）,その他金融業,TOPIX Mid400\n20260930,8572,アコム,スタンダード（内国株式）,その他金融業,TOPIX Mid400\n20260930,8591,オリックス,プライム（内国株式）,その他金融業,TOPIX Large70\n20260930,8593,三菱ＨＣキャピタル,プライム（内国株式）,その他金融業,TOPIX Mid400\n20260930,8595,ジャフコ\u3000グループ,プライム（内国株式）,証券、商品先物取引業,TOPIX Mid400\n20260930,8601,大和証券グループ本社,プライム（内国株式）,証券、商品先物取引業,TOPIX Mid400\n20260930,8604,野村ホールディングス,プライム（内国株式）,証券、商品先物取引業,TOPIX Large70\n20260930,8630,ＳＯＭＰＯホールディングス,プライム（内国株式）,保険業,TOPIX Large70\n20260930,8697,日本取引所グループ,プライム（内国株式）,その他金融業,TOPIX Large70\n20260930,8725,ＭＳ＆ＡＤインシュアランスグループホールディングス,プライム（内国株式）,保険業,TOPIX Large70\n20260930,8729,ソニーフィナンシャルグループ,プライム（内国株式）,保険業,TOPIX Core30\n20260930,8750,第一ライフグループ,プライム（内国株式）,保険業,TOPIX Large70\n20260930,8766,東京海上ホールディングス,プライム（内国株式）,保険業,TOPIX Core30\n20260930,8795,Ｔ＆Ｄホールディングス,プライム（内国株式）,保険業,TOPIX Mid400\n20260930,8801,三井不動産,プライム（内国株式）,不動産業,TOPIX Large70\n20260930,8802,三菱地所,プライム（内国株式）,不動産業,TOPIX Large70\n20260930,8804,東京建物,プライム（内国株式）,不動産業,TOPIX Mid400\n20260930,8830,住友不動産,プライム（内国株式）,不動産業,TOPIX Large70\n20260930,8876,リログループ,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,9001,東武鉄道,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9003,相鉄ホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9005,東急,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9006,京浜急行電鉄,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9007,小田急電鉄,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9008,京王電鉄,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9009,京成電鉄,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9020,東日本旅客鉄道,プライム（内国株式）,陸運業,TOPIX Large70\n20260930,9021,西日本旅客鉄道,プライム（内国株式）,陸運業,TOPIX Large70\n20260930,9022,東海旅客鉄道,プライム（内国株式）,陸運業,TOPIX Large70\n20260930,9023,東京地下鉄,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9024,西武ホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9031,西日本鉄道,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9041,近鉄グループホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9042,阪急阪神ホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9044,ＮＡＮＫＡＩ,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9045,京阪ホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9048,名古屋鉄道,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9064,ヤマトホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9065,山九,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9069,センコーグループホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9072,ニッコンホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9076,セイノーホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9101,日本郵船,プライム（内国株式）,海運業,TOPIX Large70\n20260930,9104,商船三井,プライム（内国株式）,海運業,TOPIX Large70\n20260930,9107,川崎汽船,プライム（内国株式）,海運業,TOPIX Mid400\n20260930,9142,九州旅客鉄道,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9143,ＳＧホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9147,ＮＩＰＰＯＮ\u3000ＥＸＰＲＥＳＳホールディングス,プライム（内国株式）,陸運業,TOPIX Mid400\n20260930,9201,日本航空,プライム（内国株式）,空運業,TOPIX Mid400\n20260930,9202,ＡＮＡホールディングス,プライム（内国株式）,空運業,TOPIX Large70\n20260930,9301,三菱倉庫,プライム（内国株式）,倉庫・運輸関連業,TOPIX Mid400\n20260930,9302,三井倉庫ホールディングス,プライム（内国株式）,倉庫・運輸関連業,TOPIX Mid400\n20260930,9364,上組,プライム（内国株式）,倉庫・運輸関連業,TOPIX Mid400\n20260930,9401,ＴＢＳホールディングス,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,9404,日本テレビホールディングス,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,9432,ＮＴＴ,プライム（内国株式）,情報・通信業,TOPIX Core30\n20260930,9433,ＫＤＤＩ,プライム（内国株式）,情報・通信業,TOPIX Core30\n20260930,9434,ソフトバンク,プライム（内国株式）,情報・通信業,TOPIX Core30\n20260930,9435,光通信,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,9449,ＧＭＯインターネットグループ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,9468,ＫＡＤＯＫＡＷＡ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,9501,東京電力ホールディングス,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9502,中部電力,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9503,関西電力,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9504,中国電力,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9505,北陸電力,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9506,東北電力,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9507,四国電力,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9509,北海道電力,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9513,電源開発,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9531,東京瓦斯,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9532,大阪瓦斯,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9533,東邦瓦斯,プライム（内国株式）,電気・ガス業,TOPIX Mid400\n20260930,9602,東宝,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,9616,共立メンテナンス,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,9627,アインホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,9684,スクウェア・エニックス・ホールディングス,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,9697,カプコン,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,9706,日本空港ビルデング,プライム（内国株式）,不動産業,TOPIX Mid400\n20260930,9735,セコム,プライム（内国株式）,サービス業,TOPIX Large70\n20260930,9744,メイテックグループホールディングス,プライム（内国株式）,サービス業,TOPIX Mid400\n20260930,9759,ＮＳＤ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,9766,コナミグループ,プライム（内国株式）,情報・通信業,TOPIX Mid400\n20260930,9831,ヤマダホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,9843,ニトリホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,9861,吉野家ホールディングス,プライム（内国株式）,小売業,TOPIX Mid400\n20260930,9934,因幡電機産業,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,9962,ミスミグループ本社,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,9983,ファーストリテイリング,プライム（内国株式）,小売業,TOPIX Core30\n20260930,9984,ソフトバンクグループ,プライム（内国株式）,情報・通信業,TOPIX Core30\n20260930,9987,スズケン,プライム（内国株式）,卸売業,TOPIX Mid400\n20260930,9989,サンドラッグ,プライム（内国株式）,小売業,TOPIX Mid400\n'

for rel, text in SOURCES.items():
    f = LIB / rel
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(text, encoding="utf-8")
(LIB / "data").mkdir(exist_ok=True)
(LIB / "data" / "universe_jpx.csv").write_text(UNIVERSE_CSV, encoding="utf-8")
if str(LIB.resolve()) not in sys.path:
    sys.path.insert(0, str(LIB.resolve()))

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "yfinance"], check=False)
    subprocess.run("apt-get -qq install -y fonts-ipafont-gothic > /dev/null", shell=True, check=False)
    from matplotlib import font_manager
    for p in Path("/usr/share/fonts").rglob("ipag*.ttf"):
        font_manager.fontManager.addfont(str(p))
print("準備完了:", LIB.resolve())

## 1. 設定

In [ ]:
TICKER = "8630.T"
NAME = "SOMPOホールディングス"
REFRESH = True        # 最新の株価を取り直す（Colab では毎回 True のままにする）
UPDATE_UNIVERSE = False  # True にすると JPX から最新の銘柄一覧を取り直す（通常は不要）
CAPITAL = 500_000     # 信用の保証金（円）。レバレッジ1倍で100株単位の株数を計算する
INTEREST = 0.028      # 信用の買方金利（年率）。お使いの証券会社の料率に合わせる
ENTRY_PRICE = None    # 寄りで約定した後に損切りの目安を出し直すときは買値を入れる（例: 6900.0）
RECENT_YEARS = 3      # 過去シグナル一覧に表示する年数
CHECK_CONDITIONS = True  # 5. の条件の検証（全銘柄で集計するため数分かかる）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from weekpick import data as WD
from weekpick import timing as T
from weekpick.universe import load_universe

for f in ["IPAGothic", "IPAexGothic", "Hiragino Sans", "Yu Gothic", "Meiryo", "Noto Sans CJK JP"]:
    if any(f == x.name for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f
        break
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
SIGNAL, CONDS = "wr14", T.WR_CONDITIONS
SG = T.SIGNALS[SIGNAL]

## 2. データ更新

市場全体の値動き・同業種の状況を計算するため、TOPIX500 相当の全銘柄を読み込みます。銘柄一覧は 2026/9/30 時点の JPX 一覧です。古くなったら、1. の `UPDATE_UNIVERSE` を True にしてください。

In [ ]:
if UPDATE_UNIVERSE:
    from weekpick.universe import fetch_jpx_list
    fetch_jpx_list()
u = load_universe("topix500")
if REFRESH:
    failed = WD.fetch(list(u["ticker"]), refresh=True)
    print("取得失敗:", failed or "なし")
p = WD.load_panel(list(u["ticker"]))
for n in p.notes:
    print("・", n)
last = p.close.index[-1]
print(f"最新データ: {last:%Y-%m-%d}（{'月火水木金土日'[last.weekday()]}）")
if last.weekday() != 4:
    print("⚠️ 最新データが金曜ではありません。週の途中のデータで判定しています（祝日で木曜が週最終日の場合は問題ありません）。")
sectors = u.set_index("ticker")["sector"]
ind = T.indicators(p, sectors)  # 全銘柄のパラボリックSARなどを計算するため1〜2分かかる

## 3. 今週末の判定

In [ ]:
now = T.latest(ind, [TICKER], signal=SIGNAL, conditions=CONDS)
num = ["%R14", "RSI2", "RSI14", "25日線乖離%", "週間騰落%", "市場の週間騰落%", "業種平均との差%", "同業種で%R<-80の割合%"]
now[num] = now[num].round(1)
now.insert(0, "銘柄", NAME)
now.T

In [ ]:
r = now.loc[TICKER]
if not r["シグナル"]:
    print(f"■ {NAME}: シグナルなし（%R14 = {r['%R14']}）→ 見送り")
else:
    print(f"■ {NAME}: シグナルあり（%R14 = {r['%R14']}）  参考点 {int(r['参考点']):+d}")
print("\n同時に出ている条件:")
on = [c for c in CONDS if r[c.label]]
for c in on:
    print(f"   [{c.side}] {c.label}\n        分析時点の結果: {c.note}")
if not on:
    print("   なし")
print("\n※ シグナルが出ていない週の条件は参考表示です。")

## 4. 過去のシグナル時の成績（SOMPO、2008年〜）

### 4-1. 全体と、条件ごとの上昇確率
「あり」はその条件が同時に出ていたシグナル週、「なし」は出ていなかったシグナル週です。回数が少ない（目安として20回未満）ものは、確率が大きくぶれます。

In [ ]:
hist = T.history(p, ind, [TICKER], signal=SIGNAL, conditions=CONDS)
r_ = hist["翌週騰落率"]
print(f"シグナル {len(r_)} 回 / 翌週の上昇確率 {(r_ > 0).mean():.1%} / 平均 {r_.mean():+.2%} / 中央値 {r_.median():+.2%}（配当込み・コスト控除前）")
ct = T.conditional_table(hist, CONDS).drop(columns="ticker").set_index("条件")
ct.style.format({"あり_上昇確率": "{:.1%}", "なし_上昇確率": "{:.1%}"}, na_rep="—")

### 4-2. 参考点別の過去の成績
⚠️ 条件を選んだのと同じ期間での集計です。将来もこの通りになる保証はありません。

In [ ]:
sc = hist.groupby("参考点")["翌週騰落率"].agg(回数="size", 上昇確率=lambda x: (x > 0).mean(), 平均=lambda x: x.mean())
sc.style.format({"上昇確率": "{:.1%}", "平均": "{:+.2%}"})

### 4-3. 直近のシグナル週の一覧

In [ ]:
since = last - pd.DateOffset(years=RECENT_YEARS)
recent = hist[hist["判定日"] >= since].copy()
label = {c.key: f"[{c.side}]{c.label}" for c in CONDS}
view = recent[["判定日", "買い", "売り", SG.column, "翌週騰落率", "参考点"] + list(label)].rename(columns=label)
for c in ["判定日", "買い", "売り"]:
    view[c] = view[c].dt.date
rr = recent["翌週騰落率"]
print(f"直近{RECENT_YEARS}年: {len(rr)}回, 上昇確率 {(rr > 0).mean():.1%}, 平均 {rr.mean():+.2%}")
view.style.format({SG.column: "{:.1f}", "翌週騰落率": "{:+.2%}"}).map(
    lambda v: "background-color: #e6f4ea" if v is True else "", subset=list(label.values()))

## 5. 条件の検証（参考）

各条件がシグナル週の上昇確率を本当に変えていたかを、**SOMPO・他の保険株・全銘柄**で比べます。
- 差は「条件あり − 条件なし」の上昇確率（ポイント）です。前半は2008〜2017年、後半は2018年以降です。
- 3つの比較で向きがそろい、前半・後半とも同じ向きなら、比較的信頼できます。
- 全銘柄の z値は、同じ週の銘柄を独立に数えるため、市場全体の条件では大きく出すぎます。

In [ ]:
if CHECK_CONDITIONS:
    peers = [t for t in p.close.columns if sectors.get(t) == sectors.get(TICKER) and t != TICKER]
    eff = {
        NAME: T.condition_effect(p, ind, SIGNAL, CONDS, tickers=[TICKER]),
        f"他の{sectors.get(TICKER)}（{len(peers)}銘柄）": T.condition_effect(p, ind, SIGNAL, CONDS, tickers=peers),
        "全銘柄": T.condition_effect(p, ind, SIGNAL, CONDS),
    }
    tbl = pd.concat({k: v.set_index(["条件", "向き"])[["あり_回数", "差", "前半_差", "後半_差"]] for k, v in eff.items()}, axis=1)
    display(tbl.style.format("{:+.1f}", subset=[c for c in tbl.columns if "差" in c[1]], na_rep="—"))

### 5-1. 自分で条件を追加して検証する
`ind` には日次の指標（行＝日付、列＝銘柄）が入っています。条件を True/False の表として作り、`ind["flag_名前"]` に入れて `T.Condition("名前", ...)` を作ると、上と同じ方法で検証できます。例として「RSI(14) < 30」を試します。

In [ ]:
ind["flag_my_rsi14"] = ind["rsi14"] < 30
mine = [T.Condition("my_rsi14", "RSI(14) < 30（自作の例）", "?", "")]
if CHECK_CONDITIONS:
    display(pd.concat({k: T.condition_effect(p, ind, SIGNAL, mine, tickers=tk).set_index("条件")
                       for k, tk in [(NAME, [TICKER]), ("全銘柄", None)]}, axis=1).round(3))
    print("今週の状態:", bool(ind["flag_my_rsi14"].at[last, TICKER]))

## 6. チャート（直近1年）
▲ はシグナル週の判定日です（赤＝翌週上昇、灰＝翌週下落）。

In [ ]:
start = last - pd.DateOffset(years=1)
c = p.close[TICKER].loc[start:]
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax.plot(c.index, c, color="black", lw=1.2, label="終値")
for n, col in [(25, "tab:blue"), (75, "tab:orange"), (200, "tab:green")]:
    ax.plot(c.index, p.close[TICKER].rolling(n).mean().loc[start:], lw=0.9, color=col, label=f"{n}日線")
ev = hist[hist["判定日"] >= start]
ax.scatter(ev["判定日"], c.reindex(ev["判定日"]), marker="^", s=60,
           c=np.where(ev["翌週騰落率"] > 0, "tab:red", "tab:gray"), zorder=3, label="シグナル")
ax.set_title(f"{NAME}（{TICKER}）")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.3)
ax2.plot(c.index, ind["wr14"][TICKER].loc[start:], lw=0.8, color="tab:purple")
ax2.axhline(T.WR_THRESHOLD, color="red", lw=0.8, ls="--")
ax2.set_ylim(-100, 0)
ax2.set_ylabel("%R(14)")
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 建玉と損切りの目安

**分析時点（全銘柄・SOMPO、2008〜2026/10）でのラインの比較**
- 期待値が最も高かったのは、**損切り・利確を置かずに最長1週間で決済**する形でした。
- 近いライン（損切り3〜5%、利確3〜6%）は成績を悪化させました。
- 損切りを置くなら **7%〜ATR×2** 程度の遠めに置くと、平均は1回あたり0.1〜0.2ポイント下がる一方、最悪の損失をおおむね-10%前後に抑えられました。

買値は、寄りで約定するまでは最新の終値を仮に使います。約定後は、1. の `ENTRY_PRICE` に買値を入れて再実行してください。窓を開けて損切りラインを飛び越えた場合は、ラインより不利な価格で約定します。

In [ ]:
entry = float(ENTRY_PRICE or p.close[TICKER].iloc[-1])
atr = T.trade_levels(p, TICKER, entry)["ATR14"]
lot = entry * 100
shares = int(CAPITAL // lot) * 100  # レバレッジ1倍
print(f"買値（{'入力値' if ENTRY_PRICE else '最新終値で仮置き'}）: {entry:,.1f} 円 / ATR(14): {atr:,.1f} 円（{atr / entry:.1%}）")
print(f"100株の約定代金: {lot:,.0f} 円")
if shares:
    print(f"保証金 {CAPITAL:,} 円・レバレッジ1倍での株数: {shares} 株（約定代金 {shares * entry:,.0f} 円）")
else:
    shares = 100
    print(f"⚠️ 保証金 {CAPITAL:,} 円ではレバレッジ1倍で100株を買えません。"
          f"100株ならレバレッジ約{lot / CAPITAL:.2f}倍（約定代金の{CAPITAL / lot:.0%}が保証金）になります。")
    print("   以下は100株で計算します。レバレッジをかけると、同じ値動きでも保証金に対する損益の割合が大きくなります。")
print(f"1週間（7日）持った場合の金利の目安: {shares * entry * INTEREST * 7 / 365:,.0f} 円（年率 {INTEREST:.1%} の仮定）")
rows = []
for name, stop in [("損切り7%", entry * 0.93), ("損切り ATR×2", entry - 2 * atr), ("損切り ATR×3", entry - 3 * atr)]:
    loss = (stop - entry) * shares
    rows.append({"案": name, "損切り価格": stop, "下落率": stop / entry - 1, "想定損失（円）": loss, "保証金に対して": loss / CAPITAL})
pd.DataFrame(rows).set_index("案").style.format(
    {"損切り価格": "{:,.1f}", "下落率": "{:+.1%}", "想定損失（円）": "{:+,.0f}", "保証金に対して": "{:+.1%}"})

## 8. 判断のメモ（例）

- シグナルがなければ見送る
- シグナルがあれば、参考点と 5. の検証結果を確認する（回数の少ない条件や、前半・後半で向きが違う条件は重視しない）
- 「－」の条件（株価が75日線より上＝上昇トレンド中の軽い押し目、市場全体が強い局面）が重なるときは慎重に
- 発注前に、翌週の決算発表・権利落ち・開示を各社IRやTDnetで確認する

毎週の判定と結果を記録しておくと、今後のデータで、このシグナルや条件が本当に効くかを確かめられます。